In [ ]:
# Cell 1: Imports and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import json
import os
from datetime import datetime
import time

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================
# Dataset config
# ============================================================
# Use the same dataset family XYW-Net references (RCF augmented HED-BSDS).
# This folder is created by scripts/download_xywnet_datasets.py
# Prefer the *portable* small processed dataset (easy to upload/move between devices).
SMALL_PROCESSED_ROOT = "./datasets/RCF_small/processed_HED_BSDS_small"
# If running on Kaggle, the dataset is mounted under /kaggle/input/<slug>/...
if Path("/kaggle/input").exists():
    for _p in [
        "/kaggle/input/processed-hed-bsds-small/processed_HED_BSDS_small",
        "/kaggle/input/hed-bsds-small/processed_HED_BSDS_small",
        "/kaggle/input/rcf-small/processed_HED_BSDS_small",
    ]:
        if Path(_p).exists():
            SMALL_PROCESSED_ROOT = _p
            break

# Full RCF dataset (exact upstream format; large).
RCF_HED_BSDS_ROOT = "./datasets/RCF/extracted/HED-BSDS"
RCF_VAL_RATIO = 0.10
RCF_SPLIT_SEED = 42

# Auto-select dataset mode based on what exists on disk.
# - If SMALL_PROCESSED_ROOT exists -> use ProcessedDataset (recommended for Kaggle / multi-device).
# - Else -> fall back to the full RCF list-file dataset.
if Path(SMALL_PROCESSED_ROOT).exists():
    DATASET_MODE = "processed"
    DATA_ROOT = SMALL_PROCESSED_ROOT
else:
    DATASET_MODE = "rcf_hed_bsds"
    DATA_ROOT = "./datasets/HED_Medium"  # used only when DATASET_MODE == "processed"

# Training config
BATCH_SIZE = 4
NUM_WORKERS = 0
EPOCHS_PER_VARIANT = 5  # Short training for quick ablation
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Evaluation protocol:
# - "pixel": fast proxy metric (not paper-comparable).
# - "python_bsds": boundary-matching style eval in pure Python (closest you can get without pdollar/edges).
EVAL_PROTOCOL = "python_bsds"

# ============================================================
# Organized run output folders
# ============================================================
ABLATION_DIR = Path("ablation_results")
ABLATION_DIR.mkdir(exist_ok=True)
RUNS_DIR = ABLATION_DIR / "runs"
RUNS_DIR.mkdir(exist_ok=True)

RUN_ID = datetime.now().strftime("ablation_%Y%m%d_%H%M%S")
RUN_DIR = RUNS_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

# For the rest of the notebook, treat ABLATION_DIR as the *current run* directory
ABLATION_ROOT = ABLATION_DIR
ABLATION_DIR = RUN_DIR

CHECKPOINTS_DIR = RUN_DIR / "checkpoints"
TRACED_DIR = RUN_DIR / "traced"
PREDICTIONS_DIR = RUN_DIR / "predictions"
METRICS_DIR = RUN_DIR / "metrics"
FIGURES_DIR = RUN_DIR / "figures"
for d in [CHECKPOINTS_DIR, TRACED_DIR, PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Backwards-compat variable name used elsewhere in the notebook
MODELS_DIR = CHECKPOINTS_DIR

# Prediction export controls (per variant)
EXPORT_TEST_PREDICTIONS = True
PRED_EXPORT_MAX_IMAGES = 50  # set larger if you want to export more
PRED_EXPORT_SAVE_INPUT = True
PRED_EXPORT_SAVE_GT = True

print(f"\nDATASET_MODE: {DATASET_MODE}")
if DATASET_MODE == "rcf_hed_bsds":
    print(f"RCF_HED_BSDS_ROOT: {RCF_HED_BSDS_ROOT}")
else:
    print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Epochs per variant: {EPOCHS_PER_VARIANT}")
print(f"Eval protocol: {EVAL_PROTOCOL}")
print(f"Run dir: {RUN_DIR}")
print("="*70)

Device: cuda
GPU: NVIDIA GeForce GTX 1070
Memory: 8.6 GB

Dataset: ./datasets/HED_Medium
Epochs per variant: 5
Results dir: ablation_results


In [ ]:
# Cell 2: Load Dataset
from pathlib import Path
from torch.utils.data import Dataset
import numpy as np
import cv2
import torch
import os

class ProcessedDataset(Dataset):
    """Load processed PNG images and edge maps with ImageNet normalization"""
    def __init__(self, root_dir, split='train'):
        self.root_dir = Path(root_dir)
        self.split = split
        self.img_dir = self.root_dir / split / 'images'
        self.edge_dir = self.root_dir / split / 'edges'
        
        if not self.img_dir.exists():
            raise FileNotFoundError(f"Image directory not found: {self.img_dir}")
        if not self.edge_dir.exists():
            raise FileNotFoundError(f"Edge directory not found: {self.edge_dir}")
        
        self.samples = sorted(list(self.img_dir.glob('*.png')))
        if len(self.samples) == 0:
            raise ValueError(f"No PNG files found in {self.img_dir}")
        
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        edge_path = self.edge_dir / img_path.name
        
        img = cv2.imread(str(img_path))
        if img is None:
            raise IOError(f"Failed to load image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        
        edge = cv2.imread(str(edge_path), cv2.IMREAD_GRAYSCALE)
        if edge is None:
            raise IOError(f"Failed to load edge map: {edge_path}")
        edge = edge.astype(np.float32) / 255.0
        
        img = torch.from_numpy(img).permute(2, 0, 1)
        img = (img - self.mean) / self.std
        edge = torch.from_numpy(edge).unsqueeze(0)
        
        return {'images': img, 'labels': edge, 'filename': img_path.stem}

class RCFHEDBSDSPairDataset(Dataset):
    """RCF HED-BSDS train_pair.lst dataset (augmented images + PNG edge maps)."""
    def __init__(self, root_dir, pairs, *, mean=None, std=None):
        self.root_dir = Path(root_dir)
        self.pairs = list(pairs)
        if len(self.pairs) == 0:
            raise ValueError("Empty pairs list")
        self.mean = mean if mean is not None else torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = std if std is not None else torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        rel_img, rel_gt = self.pairs[idx]
        img_path = (self.root_dir / rel_img).resolve()
        gt_path = (self.root_dir / rel_gt).resolve()

        img = cv2.imread(str(img_path))
        if img is None:
            raise IOError(f"Failed to load image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0

        edge = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
        if edge is None:
            raise IOError(f"Failed to load edge map: {gt_path}")
        edge = edge.astype(np.float32) / 255.0

        img_t = torch.from_numpy(img).permute(2, 0, 1)
        img_t = (img_t - self.mean) / self.std
        edge_t = torch.from_numpy(edge).unsqueeze(0)

        name = Path(rel_img).stem
        return {'images': img_t, 'labels': edge_t, 'filename': name}

def read_train_pair_list(root_dir):
    root_dir = Path(root_dir)
    lst = root_dir / "train_pair.lst"
    if not lst.exists():
        raise FileNotFoundError(f"Missing {lst}")
    pairs = []
    for line in lst.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 2:
            continue
        pairs.append((parts[0], parts[1]))
    return pairs

def split_pairs(pairs, val_ratio=0.1, seed=42):
    rng = np.random.RandomState(int(seed))
    idx = np.arange(len(pairs))
    rng.shuffle(idx)
    n_val = int(round(len(pairs) * float(val_ratio)))
    val_idx = set(idx[:n_val].tolist())
    train_pairs = [p for i, p in enumerate(pairs) if i not in val_idx]
    val_pairs = [p for i, p in enumerate(pairs) if i in val_idx]
    return train_pairs, val_pairs

# Load datasets
try:
    if DATASET_MODE == "rcf_hed_bsds":
        pairs = read_train_pair_list(RCF_HED_BSDS_ROOT)
        train_pairs, val_pairs = split_pairs(pairs, val_ratio=RCF_VAL_RATIO, seed=RCF_SPLIT_SEED)
        train_dataset = RCFHEDBSDSPairDataset(RCF_HED_BSDS_ROOT, train_pairs)
        val_dataset = RCFHEDBSDSPairDataset(RCF_HED_BSDS_ROOT, val_pairs)
        # NOTE: RCF HED-BSDS does not ship test GT here; use val split for evaluation metrics.
        test_dataset = val_dataset
    else:
        train_dataset = ProcessedDataset(DATA_ROOT, split='train')
        val_dataset = ProcessedDataset(DATA_ROOT, split='val') if os.path.exists(f"{DATA_ROOT}/val") else ProcessedDataset(DATA_ROOT, split='test')
        test_dataset = ProcessedDataset(DATA_ROOT, split='test')
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    
    print(f"✓ Train: {len(train_dataset)} samples")
    print(f"✓ Val:   {len(val_dataset)} samples")
    print(f"✓ Test:  {len(test_dataset)} samples")
except Exception as e:
    print(f"✗ Dataset loading failed: {e}")

✓ Train: 2016 samples
✓ Val:   432 samples
✓ Test:  432 samples


In [3]:
# Cell 3: Ablation Experiment Configurations
EXPERIMENTS = [
    # === BASELINES ===
    {'name': 'rcf_baseline',              'decoder': 'rcf', 'description': 'Standard RCF decoder (baseline)'},
    {'name': 'elc_enabled',               'decoder': 'elc', 'description': 'ELC decoder'},

    # === DISABLE ENCODER STAGES ===
    {'name': 'no_s1',                     'decoder': 'rcf', 'disable_stages': ['s1'], 'description': 'Remove stage 1 (s1o skipped)'},
    {'name': 'no_s2',                     'decoder': 'rcf', 'disable_stages': ['s2'], 'description': 'Remove stage 2 (s2o skipped)'},
    {'name': 'no_s3',                     'decoder': 'rcf', 'disable_stages': ['s3'], 'description': 'Remove stage 3 (s3o skipped)'},
    {'name': 'no_s4',                     'decoder': 'rcf', 'disable_stages': ['s4'], 'description': 'Remove stage 4 (s4o skipped)'},

    # === XYW PATHWAY ABLATIONS (disable individual) ===
    {'name': 'no_X',                      'decoder': 'rcf', 'disable_pathways': ['X'], 'description': 'Disable X (center-surround)'},
    {'name': 'no_Y',                      'decoder': 'rcf', 'disable_pathways': ['Y'], 'description': 'Disable Y (dilated contrast)'},
    {'name': 'no_W',                      'decoder': 'rcf', 'disable_pathways': ['W'], 'description': 'Disable W (directional)'},

    # === SINGLE PATHWAY BRANCHES ===
    {'name': 'only_X',                    'decoder': 'rcf', 'disable_pathways': ['Y','W'], 'description': 'Only X pathway'},
    {'name': 'only_Y',                    'decoder': 'rcf', 'disable_pathways': ['X','W'], 'description': 'Only Y pathway'},
    {'name': 'only_W',                    'decoder': 'rcf', 'disable_pathways': ['X','Y'], 'description': 'Only W pathway'},

    # === PATHWAY PAIRS ===
    {'name': 'XY_pair',                   'decoder': 'rcf', 'disable_pathways': ['W'], 'description': 'X + Y (no W)'},
    {'name': 'XW_pair',                   'decoder': 'rcf', 'disable_pathways': ['Y'], 'description': 'X + W (no Y)'},
    {'name': 'YW_pair',                   'decoder': 'rcf', 'disable_pathways': ['X'], 'description': 'Y + W (no X)'},

    # === PDC vs STANDARD CONV ===
    {'name': 'pdc_cv',                    'decoder': 'rcf', 'pdc_type': 'cv', 'description': 'Standard conv (cv) instead of PDC (2sd)'},
    {'name': 'elc_pdc_cv',                'decoder': 'elc', 'pdc_type': 'cv', 'description': 'ELC with standard conv (cv)'},

    # === NORMALIZATION VARIANTS ===
    {'name': 'norm_batch',                'decoder': 'rcf', 'norm_type': 'batch', 'description': 'BatchNorm instead of InstanceNorm'},
    {'name': 'norm_group',                'decoder': 'rcf', 'norm_type': 'group', 'description': 'GroupNorm instead of InstanceNorm'},
    {'name': 'norm_none',                 'decoder': 'rcf', 'disable_instance_norm': True, 'description': 'No normalization in decoder'},

    # === GATING & SHORTCUTS ===
    {'name': 'no_adap_gate',              'decoder': 'rcf', 'disable_adap_gate': True, 'description': 'Disable adaptive gating in refine blocks'},
    {'name': 'no_shortcuts',              'decoder': 'rcf', 'disable_shortcuts': True, 'description': 'Disable residual shortcuts'},
    {'name': 'shortcut_alpha_0.5',        'decoder': 'rcf', 'shortcut_alpha': 0.5, 'description': 'Weaken shortcuts (α=0.5)'},
    {'name': 'shortcut_alpha_2.0',        'decoder': 'rcf', 'shortcut_alpha': 2.0, 'description': 'Strengthen shortcuts (α=2.0)'},

    # === LEARNABLE DECONV ===
    {'name': 'learnable_deconv',          'decoder': 'rcf', 'learnable_deconv': True, 'description': 'Learnable bilinear deconv weights'},
    {'name': 'elc_learnable_deconv',      'decoder': 'elc', 'learnable_deconv': True, 'description': 'ELC with learnable deconv'},

    # === POOLING STRATEGY ===
    {'name': 'stride_pooling',            'decoder': 'rcf', 'pool_type': 'stride_conv', 'description': 'Stride conv (2×2) instead of max pool'},

    # === LOSS SWEEPS (Dice & Positive Weighting) ===
    {'name': 'loss_dice_0.05',            'decoder': 'rcf', 'dice_coeff': 0.05, 'description': 'Dice coeff 0.05'},
    {'name': 'loss_dice_0.1',             'decoder': 'rcf', 'dice_coeff': 0.1, 'description': 'Dice coeff 0.1'},
    {'name': 'loss_dice_0.2',             'decoder': 'rcf', 'dice_coeff': 0.2, 'description': 'Dice coeff 0.2'},
    {'name': 'loss_pos_weight_2',         'decoder': 'rcf', 'ce_pos_weight': 2.0, 'description': 'CE positive weight 2.0x'},
    {'name': 'loss_pos_weight_4',         'decoder': 'rcf', 'ce_pos_weight': 4.0, 'description': 'CE positive weight 4.0x'},
    {'name': 'loss_dice_dice_pos2',       'decoder': 'rcf', 'dice_coeff': 0.1, 'ce_pos_weight': 2.0, 'description': 'Combined: Dice 0.1 + pos weight 2x'},

    # === EVALUATION CONTROLS ===
    {'name': 'no_thinning',               'decoder': 'rcf', 'thinning': False, 'description': 'Skip edge thinning (no NMS)'},
    {'name': 'tolerance_r2',              'decoder': 'rcf', 'tolerance_radius': 2, 'description': 'GT tolerance radius r=2 (vs r=1)'},

    # === COMBINED INTERACTIONS ===
    {'name': 'rcf_batch_learn_stride',    'decoder': 'rcf', 'norm_type': 'batch', 'learnable_deconv': True, 'pool_type': 'stride_conv',
     'description': 'Combined: BatchNorm + learnable deconv + stride pooling'},
]

print(f"\n{'='*70}")
print(f"ABLATION EXPERIMENT REGISTRY: {len(EXPERIMENTS)} variants")
print(f"{'='*70}")
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"  {i:2d}. {exp['name']:30s} - {exp.get('description', '')}")


ABLATION EXPERIMENT REGISTRY: 36 variants
   1. rcf_baseline                   - Standard RCF decoder (baseline)
   2. elc_enabled                    - ELC decoder
   3. no_s1                          - Remove stage 1 (s1o skipped)
   4. no_s2                          - Remove stage 2 (s2o skipped)
   5. no_s3                          - Remove stage 3 (s3o skipped)
   6. no_s4                          - Remove stage 4 (s4o skipped)
   7. no_X                           - Disable X (center-surround)
   8. no_Y                           - Disable Y (dilated contrast)
   9. no_W                           - Disable W (directional)
  10. only_X                         - Only X pathway
  11. only_Y                         - Only Y pathway
  12. only_W                         - Only W pathway
  13. XY_pair                        - X + Y (no W)
  14. XW_pair                        - X + W (no Y)
  15. YW_pair                        - Y + W (no X)
  16. pdc_cv                         - Standard

In [4]:
# Cell 3.5: Model Architecture (XYW-Net with ablation support)
# This is the ACTUAL model - replace placeholder in Cell 6

import math

# PDC (Pixel Difference Convolution) factory
def createPDCFunc(PDC_type):
    assert PDC_type in ['cv', '2sd']
    if PDC_type == 'cv':
        return F.conv2d
    if PDC_type == '2sd':
        def func(x, weights, bias=None, stride=1, padding=0, dilation=1, groups=1):
            assert weights.size(2) == 3 and weights.size(3) == 3
            shape = weights.shape
            weights = weights.view(shape[0], shape[1], -1)
            buffer = weights.clone()
            buffer[:, :, [0, 1, 2, 3, 5, 6, 7, 8]] = buffer[:, :, [0, 1, 2, 3, 5, 6, 7, 8]] + \
                                                     buffer[:, :, [2, 7, 8, 5, 3, 0, 1, 6]] - \
                                                     2 * buffer[:, :, [1, 4, 5, 4, 4, 3, 4, 7]]
            buffer[:, :, [4]] = 0
            weights_conv = buffer.view(shape)
            return F.conv2d(x, weights_conv, bias, stride=stride, padding=padding, dilation=dilation, groups=groups)
        return func
    return F.conv2d

class Conv2d(nn.Module):
    """PDC-enabled convolution"""
    def __init__(self, pdc_type, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=False):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        self.groups = groups
        self.weight = nn.Parameter(torch.Tensor(out_channels, in_channels // groups, kernel_size, kernel_size))
        if bias:
            self.bias = nn.Parameter(torch.Tensor(out_channels))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()
        self.pdc_func = createPDCFunc(pdc_type)

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        return self.pdc_func(x, self.weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

# --- XYW components ---
class Xc1x1(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.Xcenter = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.Xcenter_relu = nn.ReLU(inplace=True)
        self.Xsurround = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, groups=in_channels)
        self.conv1_1 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.Xsurround_relu = nn.ReLU(inplace=True)
    def forward(self, x):
        xcenter = self.Xcenter_relu(self.Xcenter(x))
        xsurround = self.Xsurround_relu(self.Xsurround(x))
        xsurround = self.conv1_1(xsurround)
        return xsurround - xcenter

class Yc1x1(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.Ycenter = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.Ycenter_relu = nn.ReLU(inplace=True)
        self.Ysurround = nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=4, dilation=2, groups=in_channels)
        self.conv1_1 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.Ysurround_relu = nn.ReLU(inplace=True)
    def forward(self, x):
        ycenter = self.Ycenter_relu(self.Ycenter(x))
        ysurround = self.Ysurround_relu(self.Ysurround(x))
        ysurround = self.conv1_1(ysurround)
        return ysurround - ycenter

class W(nn.Module):
    def __init__(self, inchannel, outchannel):
        super().__init__()
        self.h = nn.Conv2d(inchannel, inchannel, kernel_size=(1, 3), padding=(0, 1), groups=inchannel)
        self.v = nn.Conv2d(inchannel, inchannel, kernel_size=(3, 1), padding=(1, 0), groups=inchannel)
        self.convh_1 = nn.Conv2d(in_channels=inchannel, out_channels=inchannel, kernel_size=1, padding=0, bias=False)
        self.convv_1 = nn.Conv2d(in_channels=inchannel, out_channels=outchannel, kernel_size=1, padding=0, bias=False)
        self.relu = nn.ReLU()
    def forward(self, x):
        h = self.relu(self.h(x)); h = self.convh_1(h)
        v = self.relu(self.v(h)); v = self.convv_1(v)
        return v

class XYW_S(nn.Module):
    def __init__(self, inchannel, outchannel, disable_pathways=None):
        super().__init__()
        self.outchannel = outchannel
        disable_pathways = set(disable_pathways or [])
        self.use_x = 'X' not in disable_pathways
        self.use_y = 'Y' not in disable_pathways
        self.use_w = 'W' not in disable_pathways
        self.y_c = Yc1x1(inchannel, outchannel) if self.use_y else None
        self.x_c = Xc1x1(inchannel, outchannel) if self.use_x else None
        self.w = W(inchannel, outchannel) if self.use_w else None
    def forward(self, x):
        xc = self.x_c(x) if self.x_c is not None else torch.zeros((x.size(0), self.outchannel, x.size(2), x.size(3)), device=x.device, dtype=x.dtype)
        yc = self.y_c(x) if self.y_c is not None else torch.zeros((x.size(0), self.outchannel, x.size(2), x.size(3)), device=x.device, dtype=x.dtype)
        w  = self.w(x)   if self.w is not None   else torch.zeros((x.size(0), self.outchannel, x.size(2), x.size(3)), device=x.device, dtype=x.dtype)
        return xc, yc, w

class XYW(nn.Module):
    def __init__(self, inchannel, outchannel, disable_pathways=None):
        super().__init__()
        disable_pathways = set(disable_pathways or [])
        self.use_x = 'X' not in disable_pathways
        self.use_y = 'Y' not in disable_pathways
        self.use_w = 'W' not in disable_pathways
        self.y_c = Yc1x1(inchannel, outchannel) if self.use_y else None
        self.x_c = Xc1x1(inchannel, outchannel) if self.use_x else None
        self.w = W(inchannel, outchannel) if self.use_w else None
    def forward(self, xc, yc, w):
        xc2 = self.x_c(xc) if self.x_c is not None else torch.zeros_like(xc)
        yc2 = self.y_c(yc) if self.y_c is not None else torch.zeros_like(yc)
        w2  = self.w(w)   if self.w is not None   else torch.zeros_like(w)
        return xc2, yc2, w2

class XYW_E(nn.Module):
    def __init__(self, inchannel, outchannel, disable_pathways=None):
        super().__init__()
        disable_pathways = set(disable_pathways or [])
        self.use_x = 'X' not in disable_pathways
        self.use_y = 'Y' not in disable_pathways
        self.use_w = 'W' not in disable_pathways
        self.y_c = Yc1x1(inchannel, outchannel) if self.use_y else None
        self.x_c = Xc1x1(inchannel, outchannel) if self.use_x else None
        self.w = W(inchannel, outchannel) if self.use_w else None
    def forward(self, xc, yc, w):
        out = 0
        if self.x_c is not None:
            out = out + self.x_c(xc)
        if self.y_c is not None:
            out = out + self.y_c(yc)
        if self.w is not None:
            out = out + self.w(w)
        if isinstance(out, int):
            # All disabled (shouldn't happen), return zeros with correct shape
            out = torch.zeros_like(xc)
        return out

# --- Encoder stages ---
class s1(nn.Module):
    def __init__(self, channel=30, disable_pathways=None, disable_shortcuts=False, shortcut_alpha=1.0):
        super().__init__()
        self.conv1 = nn.Conv2d(3, channel, kernel_size=7, padding=6, dilation=2)
        self.xyw1_1 = XYW_S(channel, channel, disable_pathways=disable_pathways)
        self.xyw1_2 = XYW(channel, channel, disable_pathways=disable_pathways)
        self.xyw1_3 = XYW_E(channel, channel, disable_pathways=disable_pathways)
        self.relu = nn.ReLU()
        self.disable_shortcuts = disable_shortcuts
        self.shortcut_alpha = float(shortcut_alpha)
    def forward(self, x):
        temp = self.relu(self.conv1(x))
        xc, yc, w = self.xyw1_1(temp)
        xc, yc, w = self.xyw1_2(xc, yc, w)
        xyw1_3 = self.xyw1_3(xc, yc, w)
        if self.disable_shortcuts:
            return xyw1_3
        return xyw1_3 + self.shortcut_alpha * temp

class s2(nn.Module):
    def __init__(self, channel=60, pool_type='maxpool', disable_pathways=None, disable_shortcuts=False, shortcut_alpha=1.0):
        super().__init__()
        self.xyw2_1 = XYW_S(channel//2, channel, disable_pathways=disable_pathways)
        self.xyw2_2 = XYW(channel, channel, disable_pathways=disable_pathways)
        self.xyw2_3 = XYW_E(channel, channel, disable_pathways=disable_pathways)
        self.shortcut = nn.Conv2d(in_channels=channel//2, out_channels=channel, kernel_size=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) if pool_type == 'maxpool' else nn.Conv2d(channel//2, channel//2, kernel_size=2, stride=2)
        self.disable_shortcuts = disable_shortcuts
        self.shortcut_alpha = float(shortcut_alpha)
    def forward(self, x):
        x = self.pool(x)
        xc, yc, w = self.xyw2_1(x)
        xc, yc, w = self.xyw2_2(xc, yc, w)
        xyw2_3 = self.xyw2_3(xc, yc, w)
        if self.disable_shortcuts:
            return xyw2_3
        shortcut = self.shortcut(x)
        return xyw2_3 + self.shortcut_alpha * shortcut

class s3(nn.Module):
    def __init__(self, channel=120, pool_type='maxpool', disable_pathways=None, disable_shortcuts=False, shortcut_alpha=1.0):
        super().__init__()
        self.xyw3_1 = XYW_S(channel//2, channel, disable_pathways=disable_pathways)
        self.xyw3_2 = XYW(channel, channel, disable_pathways=disable_pathways)
        self.xyw3_3 = XYW_E(channel, channel, disable_pathways=disable_pathways)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) if pool_type == 'maxpool' else nn.Conv2d(channel//2, channel//2, kernel_size=2, stride=2)
        self.shortcut = nn.Conv2d(in_channels=channel // 2, out_channels=channel, kernel_size=1, padding=0)
        self.disable_shortcuts = disable_shortcuts
        self.shortcut_alpha = float(shortcut_alpha)
    def forward(self, x):
        x = self.pool(x)
        if self.disable_shortcuts:
            shortcut = None
        else:
            shortcut = self.shortcut(x)
        xc, yc, w = self.xyw3_1(x)
        xc, yc, w = self.xyw3_2(xc, yc, w)
        xyw3_3 = self.xyw3_3(xc, yc, w)
        if self.disable_shortcuts:
            return xyw3_3
        return xyw3_3 + self.shortcut_alpha * shortcut

class s4(nn.Module):
    def __init__(self, channel=120, pool_type='maxpool', disable_pathways=None, disable_shortcuts=False, shortcut_alpha=1.0):
        super().__init__()
        self.xyw4_1 = XYW_S(channel, channel, disable_pathways=disable_pathways)
        self.xyw4_2 = XYW(channel, channel, disable_pathways=disable_pathways)
        self.xyw4_3 = XYW_E(channel, channel, disable_pathways=disable_pathways)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) if pool_type == 'maxpool' else nn.Conv2d(channel, channel, kernel_size=2, stride=2)
        self.shortcut = nn.Conv2d(in_channels=channel , out_channels=channel, kernel_size=1, padding=0)
        self.disable_shortcuts = disable_shortcuts
        self.shortcut_alpha = float(shortcut_alpha)
    def forward(self, x):
        x = self.pool(x)
        if self.disable_shortcuts:
            shortcut = None
        else:
            shortcut = self.shortcut(x)
        xc, yc, w = self.xyw4_1(x)
        xc, yc, w = self.xyw4_2(xc, yc, w)
        xyw4_3 = self.xyw4_3(xc, yc, w)
        if self.disable_shortcuts:
            return xyw4_3
        return xyw4_3 + self.shortcut_alpha * shortcut

class encode(nn.Module):
    def __init__(self, pdc_type='2sd', pool_type='maxpool', disable_stages=None, disable_pathways=None, disable_shortcuts=False, shortcut_alpha=1.0):
        super().__init__()
        disable_stages = set(disable_stages or [])
        self.disable_s1 = 's1' in disable_stages
        self.disable_s2 = 's2' in disable_stages
        self.disable_s3 = 's3' in disable_stages
        self.disable_s4 = 's4' in disable_stages
        self.s1_ = s1(disable_pathways=disable_pathways, disable_shortcuts=disable_shortcuts, shortcut_alpha=shortcut_alpha)
        self.s2_ = s2(pool_type=pool_type, disable_pathways=disable_pathways, disable_shortcuts=disable_shortcuts, shortcut_alpha=shortcut_alpha)
        self.s3_ = s3(pool_type=pool_type, disable_pathways=disable_pathways, disable_shortcuts=disable_shortcuts, shortcut_alpha=shortcut_alpha)
        self.s4_ = s4(pool_type=pool_type, disable_pathways=disable_pathways, disable_shortcuts=disable_shortcuts, shortcut_alpha=shortcut_alpha)
    def forward(self, x):
        s1o = self.s1_(x) if not self.disable_s1 else torch.zeros((x.size(0), 30, x.size(2), x.size(3)), device=x.device)
        s2o = self.s2_(s1o) if not self.disable_s2 else torch.zeros((x.size(0), 60, x.size(2)//2, x.size(3)//2), device=x.device)
        s3o = self.s3_(s2o) if not self.disable_s3 else torch.zeros((x.size(0), 120, x.size(2)//4, x.size(3)//4), device=x.device)
        s4o = self.s4_(s3o) if not self.disable_s4 else torch.zeros((x.size(0), 120, x.size(2)//8, x.size(3)//8), device=x.device)
        return s1o, s2o, s3o, s4o

# --- Bilinear upsample weights ---
def upsample_filt(size):
    factor = (size + 1) // 2
    center = factor - 1 if size % 2 == 1 else factor - 0.5
    og = np.ogrid[:size, :size]
    return (1 - abs(og[0] - center) / factor) * (1 - abs(og[1] - center) / factor)

def bilinear_upsample_weights(factor, number_of_classes):
    filter_size = 2 * factor - factor % 2
    weights = np.zeros((number_of_classes, number_of_classes, filter_size, filter_size), dtype=np.float32)
    upsample_kernel = upsample_filt(filter_size)
    for i in range(number_of_classes):
        weights[i, i, :, :] = upsample_kernel
    return torch.Tensor(weights)

def _make_norm(norm_type, num_channels):
    if norm_type == 'batch':
        return nn.BatchNorm2d(num_channels)
    if norm_type == 'group':
        return nn.GroupNorm(max(1, num_channels // 8), num_channels)
    if norm_type == 'none':
        return nn.Identity()
    return nn.InstanceNorm2d(num_channels)

# Decoder components
class adap_conv(nn.Module):
    def __init__(self, in_channels, out_channels, pdc_type='2sd', norm_type='instance', disable_adap_gate=False, kz=3, pd=1):
        super().__init__()
        self.conv_pdc = Conv2d(pdc_type, in_channels, out_channels, kernel_size=kz, padding=pd)
        self.norm = _make_norm(norm_type, out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.weight = nn.Parameter(torch.Tensor([0.5])) if not disable_adap_gate else None
    def forward(self, x):
        x = self.conv_pdc(x)
        x = self.norm(x)
        x = self.relu(x)
        if self.weight is not None:
            x = x * self.weight.sigmoid()
        return x

class Refine_block2_1(nn.Module):
    def __init__(self, in_channel, out_channel, factor, pdc_type='2sd', norm_type='instance', disable_adap_gate=False, require_grad=False):
        super().__init__()
        self.pre_conv1 = adap_conv(in_channel[0], out_channel, pdc_type, norm_type, disable_adap_gate, kz=3, pd=1)
        self.pre_conv2 = adap_conv(in_channel[1], out_channel, pdc_type, norm_type, disable_adap_gate, kz=3, pd=1)
        self.factor = factor
        self.deconv_weight = nn.Parameter(bilinear_upsample_weights(factor, out_channel), requires_grad=require_grad)
    def forward(self, x1, x2):
        x1 = self.pre_conv1(x1)
        x2 = self.pre_conv2(x2)
        x2 = F.conv_transpose2d(x2, self.deconv_weight, stride=self.factor, padding=int(self.factor/2),
                                output_padding=(x1.size(2) - x2.size(2)*self.factor, x1.size(3) - x2.size(3)*self.factor))
        return x1 + x2

class decode_rcf(nn.Module):
    def __init__(self, pdc_type='2sd', norm_type='instance', disable_adap_gate=False, require_grad=False):
        super().__init__()
        self.f43 = Refine_block2_1(in_channel=(120, 120), out_channel=60, factor=2, pdc_type=pdc_type, norm_type=norm_type, disable_adap_gate=disable_adap_gate, require_grad=require_grad)
        self.f32 = Refine_block2_1(in_channel=(60, 60), out_channel=30, factor=2, pdc_type=pdc_type, norm_type=norm_type, disable_adap_gate=disable_adap_gate, require_grad=require_grad)
        self.f21 = Refine_block2_1(in_channel=(30, 30), out_channel=24, factor=2, pdc_type=pdc_type, norm_type=norm_type, disable_adap_gate=disable_adap_gate, require_grad=require_grad)
        self.f = nn.Conv2d(24, 1, kernel_size=1, padding=0)
    def forward(self, x):
        s3 = self.f43(x[2], x[3])
        s2 = self.f32(x[1], s3)
        s1 = self.f21(x[0], s2)
        out = self.f(s1)
        return out.sigmoid()

class ELCBlock(nn.Module):
    """Edge Localization Convolution head (logits)"""
    def __init__(self, ch, pdc_type='2sd', norm_type='instance'):
        super().__init__()
        self.pdc = Conv2d(pdc_type, ch, ch, 3, padding=1)
        self.norm = _make_norm(norm_type, ch)
        self.relu = nn.ReLU(inplace=True)
        self.out = nn.Conv2d(ch, 1, 1)
    def forward(self, x):
        x = self.pdc(x)
        x = self.norm(x)
        x = self.relu(x)
        return self.out(x)

class decode_elc(nn.Module):
    """RCF pyramid + ELC head; returns probability map"""
    def __init__(self, pdc_type='2sd', norm_type='instance', disable_adap_gate=False, require_grad=False):
        super().__init__()
        self.f43 = Refine_block2_1(in_channel=(120, 120), out_channel=60, factor=2, pdc_type=pdc_type, norm_type=norm_type, disable_adap_gate=disable_adap_gate, require_grad=require_grad)
        self.f32 = Refine_block2_1(in_channel=(60, 60), out_channel=30, factor=2, pdc_type=pdc_type, norm_type=norm_type, disable_adap_gate=disable_adap_gate, require_grad=require_grad)
        self.f21 = Refine_block2_1(in_channel=(30, 30), out_channel=24, factor=2, pdc_type=pdc_type, norm_type=norm_type, disable_adap_gate=disable_adap_gate, require_grad=require_grad)
        self.elc = ELCBlock(24, pdc_type=pdc_type, norm_type=norm_type)
    def forward(self, x):
        s3 = self.f43(x[2], x[3])
        s2 = self.f32(x[1], s3)
        s1 = self.f21(x[0], s2)
        logits = self.elc(s1)
        return logits.sigmoid()

class XYWNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        decoder = config.get('decoder', 'rcf')
        pdc_type = config.get('pdc_type', '2sd')
        pool_type = config.get('pool_type', 'maxpool')
        disable_stages = config.get('disable_stages', [])
        disable_pathways = config.get('disable_pathways', [])
        norm_type = config.get('norm_type', 'instance')
        if config.get('disable_instance_norm', False):
            norm_type = 'none'
        disable_adap_gate = config.get('disable_adap_gate', False)
        disable_shortcuts = config.get('disable_shortcuts', False)
        shortcut_alpha = config.get('shortcut_alpha', 1.0)
        learnable_deconv = config.get('learnable_deconv', False)
        self.encode = encode(pdc_type=pdc_type, pool_type=pool_type, disable_stages=disable_stages,
                            disable_pathways=disable_pathways, disable_shortcuts=disable_shortcuts,
                            shortcut_alpha=shortcut_alpha)
        if decoder == 'rcf':
            self.decode = decode_rcf(pdc_type=pdc_type, norm_type=norm_type,
                                     disable_adap_gate=disable_adap_gate, require_grad=learnable_deconv)
        elif decoder == 'elc':
            self.decode = decode_elc(pdc_type=pdc_type, norm_type=norm_type,
                                     disable_adap_gate=disable_adap_gate, require_grad=learnable_deconv)
        else:
            raise NotImplementedError(f"Decoder {decoder} not implemented")
    def forward(self, x):
        end_points = self.encode(x)
        return self.decode(end_points)

print("✓ Model architecture loaded with ablation support (RCF + ELC)")

✓ Model architecture loaded with ablation support (RCF + ELC)


In [ ]:
# Cell 4: Results Tracker & Aggregation
class AblationTracker:
    """Track, log, and analyze ablation results"""
    def __init__(self, save_dir):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(exist_ok=True)
        self.results = []

    def log_experiment(self, exp_name, config, ods, ois, ap, train_loss, epoch, elapsed_time):
        """Log results for one variant"""
        effective_norm = 'none' if config.get('disable_instance_norm', False) else config.get('norm_type', 'instance')
        entry = {
            'experiment': exp_name,
            'decoder': config.get('decoder', 'rcf'),
            'disable_stages': ','.join(config.get('disable_stages', [])) or 'none',
            'disable_pathways': ','.join(config.get('disable_pathways', [])) or 'none',
            'pdc_type': config.get('pdc_type', '2sd'),
            'norm_type': effective_norm,
            'disable_adap_gate': config.get('disable_adap_gate', False),
            'disable_shortcuts': config.get('disable_shortcuts', False),
            'shortcut_alpha': config.get('shortcut_alpha', 1.0),
            'learnable_deconv': config.get('learnable_deconv', False),
            'pool_type': config.get('pool_type', 'maxpool'),
            'dice_coeff': config.get('dice_coeff', 0.0),
            'ce_pos_weight': config.get('ce_pos_weight', 1.0),
            'ODS': ods,
            'OIS': ois,
            'AP': ap,
            'train_loss': train_loss,
            'best_epoch': epoch,
            'time_sec': elapsed_time,
        }
        self.results.append(entry)
        return entry

    def save_csv(self):
        """Save results to CSV"""
        df = pd.DataFrame(self.results)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        csv_path = self.save_dir / f"ablation_results_{timestamp}.csv"
        df.to_csv(csv_path, index=False)
        print(f"\n✓ Saved results CSV: {csv_path}")
        return df

    def plot_comparison(self, metric='ODS', top_n=20):
        """Plot metric across all variants (top N)"""
        if not self.results:
            print("No results to plot")
            return
        df = pd.DataFrame(self.results)
        df_top = df.nlargest(top_n, metric)
        fig, ax = plt.subplots(figsize=(12, 10))
        colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(df_top)))
        ax.barh(df_top['experiment'], df_top[metric], color=colors)
        ax.set_xlabel(metric, fontsize=11)
        ax.set_title(f'Ablation Study: Top {top_n} Variants by {metric}', fontsize=13, fontweight='bold')
        ax.invert_yaxis()
        plt.tight_layout()
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        plot_path = self.save_dir / f"ablation_top{top_n}_{metric}_{timestamp}.png"
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved plot: {plot_path}")
        plt.show()
        return df_top

    def summary_table(self, top_n=10):
        """Print summary table of top variants"""
        if not self.results:
            print("No results to summarize")
            return
        df = pd.DataFrame(self.results)
        print(f"\n{'='*80}")
        print(f"TOP {top_n} VARIANTS (by ODS)")
        print(f"{'='*80}")
        df_top = df.nlargest(top_n, 'ODS')[['experiment', 'ODS', 'OIS', 'AP', 'train_loss', 'time_sec']]
        print(df_top.to_string(index=False))
        print(f"\n{'='*80}")
        print(f"WORST {top_n} VARIANTS (by ODS) - BIGGEST DROPS")
        print(f"{'='*80}")
        df_bottom = df.nsmallest(top_n, 'ODS')[['experiment', 'ODS', 'OIS', 'AP', 'train_loss', 'time_sec']]
        print(df_bottom.to_string(index=False))

tracker = AblationTracker(RUN_DIR)
print("✓ Tracker initialized")

✓ Tracker initialized


In [ ]:
# Cell 5: Loss & Metrics Functions

# NOTE (IMPORTANT):
# - The "pixelwise" ODS/OIS/AP below is a fast proxy metric and is NOT paper-comparable.
# - Paper-standard ODS/OIS for BSDS-style edge detection is computed with the BSDS boundary benchmark
#   implementation from pdollar/edges (as referenced by the upstream XYW-Net repo).
# - This notebook now includes an optional "bsds" evaluation path that exports predictions + builds
#   BSDS-style groundTruth .mat files (from your GT edge PNGs) and then runs pdollar/edges via MATLAB/Octave.

import shutil
import subprocess
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn as nn

DICE_COEFF = 0.0  # Default; overridden per variant
CE_POS_WEIGHT = 1.0  # Default; overridden per variant
THINNING = True  # Default
TOLERANCE_RADIUS = 1  # Default

# Evaluation protocol:
# - "pixel": fast proxy metric (NOT paper-comparable).
# - "bsds": pdollar/edges boundary benchmark (paper-standard matching). Requires MATLAB or Octave + pdollar/edges.
EVAL_PROTOCOL = os.environ.get("EDGE_EVAL_PROTOCOL", "pixel").strip().lower()
PDOLLAR_EDGES_DIR = os.environ.get("PDOLLAR_EDGES_DIR", "").strip()  # path to pdollar/edges toolbox root

class EdgeLoss(nn.Module):
    def __init__(self, dice_coeff=0.0, ce_pos_weight=1.0):
        super().__init__()
        self.eps = 1e-6
        self.dice_coeff = float(dice_coeff)
        self.ce_pos_weight = float(ce_pos_weight)

    def _cross_entropy_with_weight(self, pred, labels):
        # pred, labels: (H, W)
        p = pred.view(-1).clamp(self.eps, 1.0 - self.eps)
        y = labels.view(-1)
        pos = y > 0
        neg = y == 0
        p_pos = p[pos]
        p_neg = p[neg]
        w_pos = y[pos] * self.ce_pos_weight  # annotation strength * optional multiplier
        loss = 0.0
        if p_pos.numel() > 0:
            loss = loss + (-p_pos.log() * w_pos).mean()
        if p_neg.numel() > 0:
            loss = loss + (-(1.0 - p_neg).log()).mean()
        return loss

    def _dice_loss(self, pred, labels):
        p = pred.view(-1)
        y = labels.view(-1)
        num = (p * y).sum() * 2 + self.eps
        den = p.sum() + y.sum() + self.eps
        # Keep same form as your main notebook (inverse dice)
        return (num / den).pow(-1)

    def forward(self, pred, labels):
        # pred, labels: (B,1,H,W) probability maps
        B = pred.shape[0]
        total_ce = 0.0
        total_dice = 0.0
        for i in range(B):
            total_ce = total_ce + self._cross_entropy_with_weight(pred[i, 0], labels[i, 0])
            total_dice = total_dice + self._dice_loss(pred[i, 0], labels[i, 0])
        ce = total_ce / max(B, 1)
        dc = total_dice / max(B, 1)
        return ce + self.dice_coeff * dc

def nms_edge(pred, apply_thinning=True):
    """Edge thinning via ximgproc or Canny fallback."""
    if not apply_thinning:
        return pred
    pred_blur = cv2.GaussianBlur(pred, (3, 3), 0)
    pred_uint8 = (np.clip(pred_blur, 0, 1) * 255).astype(np.uint8)
    try:
        if hasattr(cv2, 'ximgproc') and hasattr(cv2.ximgproc, 'thinning'):
            edges = cv2.ximgproc.thinning(pred_uint8, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)
        else:
            raise AttributeError
    except Exception:
        edges = cv2.Canny(pred_uint8, 50, 150)
    return edges.astype(np.float32) / 255.0

def dilate_gt(gt, r=1):
    """Dilate GT for tolerance matching (pixelwise proxy eval only)."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    return cv2.dilate(gt, kernel)

def compute_pixelwise_ods_ois_ap(preds, labels, apply_thinning=True, tolerance_radius=1, thresholds=30):
    """Fast proxy: pixelwise F1 sweep (NOT paper-standard BSDS boundary matching)."""
    from sklearn.metrics import average_precision_score
    threshs = np.linspace(0.05, 0.95, thresholds)
    all_preds = []
    all_labels = []
    ois_f1_scores = []
    for pred, label in zip(preds, labels):
        pred = nms_edge(pred, apply_thinning=apply_thinning)
        label_binary = (label > 0.5).astype(np.float32)
        label_tol = dilate_gt(label_binary, r=tolerance_radius).flatten()
        pred_smooth = cv2.GaussianBlur(pred, (3, 3), 0).flatten()
        all_preds.append(pred_smooth)
        all_labels.append(label_tol)
        best_f1 = 0.0
        for t in threshs:
            pred_bin = (pred_smooth >= t).astype(np.float32)
            tp = np.sum(pred_bin * label_tol)
            fp = np.sum(pred_bin * (1 - label_tol))
            fn = np.sum((1 - pred_bin) * label_tol)
            precision = tp / (tp + fp + 1e-8)
            recall = tp / (tp + fn + 1e-8)
            f1 = 2 * precision * recall / (precision + recall + 1e-8)
            best_f1 = max(best_f1, f1)
        ois_f1_scores.append(best_f1)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    best_ods = 0.0
    for t in threshs:
        pred_bin = (all_preds >= t).astype(np.float32)
        tp = np.sum(pred_bin * all_labels)
        fp = np.sum(pred_bin * (1 - all_labels))
        fn = np.sum((1 - pred_bin) * all_labels)
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        best_ods = max(best_ods, f1)
    ois = float(np.mean(ois_f1_scores))
    try:
        ap = float(average_precision_score(all_labels, all_preds))
    except Exception:
        ap = 0.0
    return best_ods, ois, ap

def _require_scipy_savemat():
    try:
        from scipy.io import savemat  # type: ignore
        return savemat
    except Exception as e:
        raise ImportError(
            "SciPy is required for BSDS groundTruth .mat writing. Install with: pip install scipy"
        ) from e

def write_bsds_groundtruth_mats_from_arrays(edge_maps, names, out_dir):
    """Create BSDS-style `groundTruth` .mat files from edge maps.

    This enables using pdollar/edges boundary benchmark code even when your GT is a single edge PNG (not the
    original BSDS multi-annotator GT). The matching algorithm will be paper-standard, but results may still differ
    from the paper if the dataset/GT differ.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    savemat = _require_scipy_savemat()
    if len(edge_maps) != len(names):
        raise ValueError("edge_maps and names must have same length")
    for edge, name in zip(edge_maps, names):
        bnd = (edge > 0.5).astype(np.uint8)
        seg = np.zeros_like(bnd, dtype=np.uint16)
        gt_cell = np.empty((1, 1), dtype=object)
        gt_cell[0, 0] = {"Boundaries": bnd, "Segmentation": seg}
        savemat(out_dir / f"{name}.mat", {"groundTruth": gt_cell})
    return out_dir

def export_pred_pngs_for_edges(pred_maps, names, out_dir):
    """Export predictions as uint8 PNGs for pdollar/edges eval."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    if len(pred_maps) != len(names):
        raise ValueError("pred_maps and names must have same length")
    for pred, name in zip(pred_maps, names):
        pred_uint8 = (np.clip(pred, 0, 1) * 255).astype(np.uint8)
        cv2.imwrite(str(out_dir / f"{name}.png"), pred_uint8)
    return out_dir

def _find_matlab_engine(engine="auto"):
    engine = (engine or "auto").strip().lower()
    if engine != "auto":
        return engine
    if shutil.which("matlab") is not None:
        return "matlab"
    if shutil.which("octave") is not None:
        return "octave"
    return ""

def run_pdollar_edges_eval(res_dir, gt_dir, out_dir, edges_dir=PDOLLAR_EDGES_DIR, engine="auto"):
    """Run pdollar/edges BSDS boundary evaluation via MATLAB or Octave.

    Requirements:
    - `edges_dir` points to pdollar/edges toolbox root (must contain `edgesEvalDir.m` on path).
    - `gt_dir` contains BSDS-style `groundTruth/*.mat` files (name-matched to your result PNGs).
    """
    edges_dir = str(edges_dir).strip()
    if not edges_dir:
        raise ValueError("PDOLLAR_EDGES_DIR is not set. Point it to the pdollar/edges toolbox root.")
    res_dir = Path(res_dir)
    gt_dir = Path(gt_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    engine = _find_matlab_engine(engine)
    if not engine:
        raise RuntimeError("Neither `matlab` nor `octave` found on PATH. Install MATLAB or GNU Octave.")

    # Write a small runner script (kept in out_dir for inspection).
    runner = out_dir / "run_edges_eval.m"
    runner.write_text("\n".join([
        "try",
        f"  addpath(genpath('{edges_dir.replace('\\', '/')}'));",
        f"  resDir='{str(res_dir).replace('\\', '/')}';",
        f"  gtDir='{str(gt_dir).replace('\\', '/')}';",
        f"  outDir='{str(out_dir).replace('\\', '/')}';",
        "  if exist('edgesEvalDir','file')==2",
        "    try",
        "      edgesEvalDir(resDir, gtDir, outDir);",
        "    catch",
        "      edgesEvalDir(resDir, gtDir);",
        "    end",
        "  elseif exist('edgesEval','file')==2",
        "    edgesEval(resDir, gtDir, outDir);",
        "  else",
        "    error('pdollar/edges functions not found on path (edgesEvalDir / edgesEval).');",
        "  end",
        "catch ME",
        "  disp(getReport(ME,'extended'));",
        "  exit(1);",
        "end",
        "exit(0);",
    ]), encoding="utf-8")

    if engine == "matlab":
        cmd = ["matlab", "-batch", f"run('{str(runner).replace('\\', '/')}')"]
    else:
        # Octave
        cmd = ["octave", "--no-gui", "--quiet", str(runner)]

    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"pdollar/edges eval failed (rc={proc.returncode}).\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}")
    return out_dir

def parse_pdollar_eval_bdry(out_dir):
    """Parse `eval_bdry.txt` output from pdollar/edges.

    Typical format is 8 floats:
    [thr, ODS_R, ODS_P, ODS_F, OIS_R, OIS_P, OIS_F, AP]
    """
    out_dir = Path(out_dir)
    p = out_dir / "eval_bdry.txt"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. pdollar/edges may have written to a different location.")
    parts = p.read_text(encoding="utf-8").strip().split()
    vals = [float(x) for x in parts]
    if len(vals) >= 8:
        return {
            "thr": vals[0],
            "ODS": vals[3],
            "OIS": vals[6],
            "AP": vals[7],
            "raw": vals,
        }
    return {"raw": vals}

print("✓ Loss & metrics loaded: pixelwise proxy + optional pdollar/edges BSDS eval")

✓ Loss & metrics functions loaded (EdgeLoss + per-image eval)


In [ ]:
# Cell 6: Training Function for a Single Variant

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def _denorm_to_uint8(img_chw: torch.Tensor) -> np.ndarray:
    """Convert normalized CHW tensor to uint8 RGB HWC."""
    x = img_chw.detach().cpu()
    x = x * IMAGENET_STD + IMAGENET_MEAN
    x = x.clamp(0.0, 1.0)
    x = (x.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return x

def _save_gray01_png(path: Path, arr01: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    a = np.clip(arr01, 0.0, 1.0)
    cv2.imwrite(str(path), (a * 255.0).round().astype(np.uint8))

@torch.no_grad()
def export_prediction_triplets(model, loader, out_dir: Path, *, max_images: int = 50, save_input: bool = True, save_gt: bool = True):
    """Export per-image predictions (and optionally input/GT) for a subset of a loader."""
    out_dir = Path(out_dir)
    pred_dir = out_dir / "pred"
    inp_dir = out_dir / "input"
    gt_dir = out_dir / "gt"
    pred_dir.mkdir(parents=True, exist_ok=True)
    if save_input:
        inp_dir.mkdir(parents=True, exist_ok=True)
    if save_gt:
        gt_dir.mkdir(parents=True, exist_ok=True)

    manifest = []
    exported = 0
    model.eval()
    for batch in loader:
        images = batch['images'].to(DEVICE)
        labels = batch.get('labels', None)
        filenames = batch.get('filename', ['unknown'] * images.shape[0])
        outputs = model(images)
        bsz = outputs.shape[0]
        for i in range(bsz):
            if exported >= max_images:
                df = pd.DataFrame(manifest)
                df.to_csv(out_dir / "manifest.csv", index=False)
                return df
            name = str(filenames[i])
            safe = "".join([c if c.isalnum() or c in ('-', '_') else '_' for c in name])
            pred = outputs[i, 0].detach().cpu().numpy().astype(np.float32)
            _save_gray01_png(pred_dir / f"{safe}.png", pred)
            if save_gt and labels is not None:
                gt = labels[i, 0].detach().cpu().numpy().astype(np.float32)
                _save_gray01_png(gt_dir / f"{safe}.png", gt)
            if save_input:
                rgb = _denorm_to_uint8(batch['images'][i])
                bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
                cv2.imwrite(str(inp_dir / f"{safe}.png"), bgr)
            manifest.append({"name": name, "pred": f"pred/{safe}.png", "gt": f"gt/{safe}.png" if save_gt and labels is not None else "", "input": f"input/{safe}.png" if save_input else ""})
            exported += 1
    df = pd.DataFrame(manifest)
    df.to_csv(out_dir / "manifest.csv", index=False)
    return df

def train_variant(model_name, config, epochs=EPOCHS_PER_VARIANT):
    """Train a single ablation variant and save organized artifacts under RUN_DIR."""
    print(f"\n{'='*70}")
    print(f"Variant: {model_name}")
    print(f"Epochs: {epochs}")
    print(f"Config: {config}")
    print(f"{'='*70}")

    # Per-variant output folders
    variant_dir = RUN_DIR / "variants" / model_name
    variant_ckpt_dir = variant_dir / "checkpoints"
    variant_traced_dir = variant_dir / "traced"
    variant_pred_dir = variant_dir / "predictions"
    for d in [variant_ckpt_dir, variant_traced_dir, variant_pred_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # Optional dual backup: local folder + S3/S3-compatible.
    # - Local backup: set env var XYW_BACKUP_DIR to a writable folder.
    # - S3 backup: set env var XYW_S3_BUCKET (+ optional XYW_S3_PREFIX / XYW_S3_ENDPOINT_URL).
    import shutil

    BACKUP_BASE = os.environ.get("XYW_BACKUP_DIR", "").strip()
    VARIANT_BACKUP_DIR = None
    if BACKUP_BASE:
        VARIANT_BACKUP_DIR = Path(BACKUP_BASE) / RUN_DIR.name / "variants" / model_name
        VARIANT_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        print(f"(local backup enabled) {VARIANT_BACKUP_DIR}")
    else:
        print("(local backup disabled) Set env var XYW_BACKUP_DIR to enable")

    S3_BUCKET = os.environ.get("XYW_S3_BUCKET", "").strip() or None
    S3_PREFIX = os.environ.get("XYW_S3_PREFIX", "").strip()
    S3_ENDPOINT_URL = os.environ.get("XYW_S3_ENDPOINT_URL", "").strip() or None

    S3_CLIENT = None
    if S3_BUCKET:
        try:
            import boto3
            S3_CLIENT = boto3.client("s3", endpoint_url=S3_ENDPOINT_URL)
            print(f"(S3 backup enabled) bucket={S3_BUCKET} prefix={S3_PREFIX or '(none)'} endpoint={S3_ENDPOINT_URL or '(aws)'}")
        except Exception as e:
            S3_CLIENT = None
            print(f"! S3 backup disabled (boto3/config issue): {e}")
    else:
        print("(S3 backup disabled) Set env var XYW_S3_BUCKET to enable")

    def _backup_rel_for(path: Path) -> Path:
        try:
            return path.relative_to(variant_dir)
        except Exception:
            return Path("extra") / path.name

    def _backup_put(path: Path) -> None:
        path = Path(path)
        if not path.exists():
            return
        rel = _backup_rel_for(path)

        # Local mirror
        if VARIANT_BACKUP_DIR is not None:
            dst = VARIANT_BACKUP_DIR / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            tmp = Path(str(dst) + ".tmp")
            try:
                shutil.copy2(str(path), str(tmp))
                os.replace(str(tmp), str(dst))
            except Exception as e:
                print(f"! Local backup failed for {rel}: {e}")
                try:
                    if tmp.exists():
                        tmp.unlink()
                except Exception:
                    pass

        # S3 mirror
        if S3_CLIENT is not None and S3_BUCKET is not None:
            key_parts = []
            if S3_PREFIX:
                key_parts.append(S3_PREFIX.strip("/"))
            key_parts.extend([RUN_DIR.name, "variants", model_name, rel.as_posix()])
            key = "/".join(key_parts)
            try:
                S3_CLIENT.upload_file(str(path), S3_BUCKET, key)
            except Exception as e:
                print(f"! S3 upload failed for {rel}: {e}")

    # Save variant config
    config_path = variant_dir / "config.json"
    config_path.write_text(json.dumps({
        "model_name": model_name,
        "config": config,
        "dataset_mode": DATASET_MODE,
        "rcf_root": RCF_HED_BSDS_ROOT if DATASET_MODE == "rcf_hed_bsds" else "",
        "data_root": DATA_ROOT if DATASET_MODE == "processed" else "",
        "epochs": int(epochs),
        "batch_size": int(BATCH_SIZE),
        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),
        "eval_protocol": config.get("eval_protocol", EVAL_PROTOCOL),
        "created_at": datetime.now().isoformat(),
    }, indent=2), encoding="utf-8")
    _backup_put(config_path)

    # Build ablated model
    model = XYWNet(config).to(DEVICE)

    # Loss and optimizer (aligned with full notebook)
    criterion = EdgeLoss(
        dice_coeff=config.get('dice_coeff', 0.0),
        ce_pos_weight=config.get('ce_pos_weight', 1.0),
    )
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    # Tracking
    epoch_losses = []
    val_curve = []
    best_ods = 0.0
    best_epoch = 0
    example_input = None

    # Persist per-epoch artifacts (crash-safe).
    # This is the most reliable way to run on unstable/remote machines: each epoch leaves a checkpoint + metrics on disk.
    import csv

    epoch_metrics_csv = variant_dir / "epoch_metrics.csv"

    def _atomic_write_text(path: Path, text: str) -> None:
        tmp = Path(str(path) + '.tmp')
        tmp.write_text(text, encoding='utf-8')
        os.replace(str(tmp), str(path))
        _backup_put(path)

    def _atomic_torch_save(obj, path: Path) -> None:
        tmp = Path(str(path) + '.tmp')
        torch.save(obj, str(tmp))
        os.replace(str(tmp), str(path))
        _backup_put(path)

    def _append_epoch_row(row: dict) -> None:
        is_new = not epoch_metrics_csv.exists()
        with open(epoch_metrics_csv, 'a', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(row.keys()))
            if is_new:
                w.writeheader()
            w.writerow(row)
            f.flush()
            try:
                os.fsync(f.fileno())
            except Exception:
                pass
        _backup_put(epoch_metrics_csv)

    # Save a full checkpoint each epoch (default True for reliability; can be disabled per-variant if needed).
    SAVE_EACH_EPOCH_CHECKPOINT = bool(config.get("save_each_epoch_checkpoint", True))

    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            images = batch['images'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            if example_input is None:
                example_input = images[:1].detach()

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += float(loss.item())

        epoch_loss /= max(len(train_loader), 1)
        epoch_losses.append(epoch_loss)

        # Validate (FAST proxy metric; not paper-comparable)
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for batch in val_loader:
                images = batch['images'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(images)
                for i in range(outputs.shape[0]):
                    val_preds.append(outputs[i, 0].detach().cpu().numpy().astype(np.float32))
                    val_labels.append(labels[i, 0].detach().cpu().numpy().astype(np.float32))

        ods, ois, ap = compute_pixelwise_ods_ois_ap(
            val_preds,
            val_labels,
            apply_thinning=config.get('thinning', True),
            tolerance_radius=config.get('tolerance_radius', 1),
            thresholds=30,
        )
        val_curve.append({"epoch": int(epoch + 1), "loss": float(epoch_loss), "proxyODS": float(ods), "proxyOIS": float(ois), "proxyAP": float(ap)})

        # Save per-epoch full checkpoint
        if SAVE_EACH_EPOCH_CHECKPOINT:
            _atomic_torch_save({
                "epoch": int(epoch + 1),
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_ods": float(best_ods),
                "config": config,
            }, variant_ckpt_dir / f"epoch_{epoch+1:03d}_full.pth")

        if ods > best_ods:
            best_ods = float(ods)
            best_epoch = epoch + 1
            best_epoch_name = f"{model_name}_epoch{best_epoch}.pth"

            # Save best weights (variant-local)
            _atomic_torch_save(model.state_dict(), variant_ckpt_dir / best_epoch_name)
            _atomic_torch_save(model.state_dict(), variant_ckpt_dir / "best_weights.pth")
            _atomic_torch_save({
                "epoch": int(best_epoch),
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_ods": float(best_ods),
                "config": config,
            }, variant_ckpt_dir / "best_full_checkpoint.pth")

            # Also save a copy in the run-level checkpoints folder for easy browsing/compatibility
            _atomic_torch_save(model.state_dict(), MODELS_DIR / best_epoch_name)
            _atomic_torch_save(model.state_dict(), MODELS_DIR / f"{model_name}_best.pth")

            # Export traced model (.pt) for the best checkpoint
            if example_input is not None:
                try:
                    traced = torch.jit.trace(model, example_input)
                    traced_path = variant_traced_dir / "best_model.pt"
                    traced.save(str(traced_path))
                    _backup_put(traced_path)
                except Exception as e:
                    print(f"! Trace failed for {model_name}: {e}")

        # Persist epoch metrics + history snapshot (so you can recover even if runtime stops).
        _append_epoch_row({
            'epoch': int(epoch + 1),
            'train_loss': float(epoch_loss),
            'val_proxyODS': float(ods),
            'val_proxyOIS': float(ois),
            'val_proxyAP': float(ap),
            'best_proxyODS': float(best_ods),
            'best_epoch': int(best_epoch),
        })
        _atomic_write_text(variant_dir / 'history.json', json.dumps({'epoch_losses': epoch_losses, 'val_curve': val_curve}, indent=2))

        scheduler.step()
        print(f"  Epoch {epoch+1}: Loss={epoch_loss:.4f} | proxyODS={ods:.4f} | proxyOIS={ois:.4f} | proxyAP={ap:.4f}")

    elapsed = time.time() - start_time

    # Save last checkpoint + curves
    _atomic_torch_save({
        "epoch": int(epochs),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_ods": float(best_ods),
        "best_epoch": int(best_epoch),
        "config": config,
    }, variant_ckpt_dir / "last_full_checkpoint.pth")
    _atomic_write_text(variant_dir / 'history.json', json.dumps({'epoch_losses': epoch_losses, 'val_curve': val_curve}, indent=2))

    # Final test evaluation
    model.eval()
    test_preds = []
    test_labels = []
    test_names = []
    with torch.no_grad():
        for batch in test_loader:
            images = batch['images'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(images)
            filenames = batch.get('filename', ['unknown'] * outputs.shape[0])
            for i in range(outputs.shape[0]):
                test_preds.append(outputs[i, 0].detach().cpu().numpy().astype(np.float32))
                test_labels.append(labels[i, 0].detach().cpu().numpy().astype(np.float32))
                test_names.append(str(filenames[i]))

    # Always compute proxy metric (fast)
    final_ods, final_ois, final_ap = compute_pixelwise_ods_ois_ap(
        test_preds,
        test_labels,
        apply_thinning=config.get('thinning', True),
        tolerance_radius=config.get('tolerance_radius', 1),
        thresholds=30,
    )

    # Save test proxy metrics
    test_proxy_path = variant_dir / "test_metrics_proxy.json"
    test_proxy_path.write_text(json.dumps({
        "proxyODS": float(final_ods),
        "proxyOIS": float(final_ois),
        "proxyAP": float(final_ap),
        "best_epoch": int(best_epoch),
        "elapsed_sec": float(elapsed),
    }, indent=2), encoding="utf-8")
    _backup_put(test_proxy_path)

    # Export test predictions (subset)
    if EXPORT_TEST_PREDICTIONS:
        try:
            export_prediction_triplets(
                model,
                test_loader,
                variant_pred_dir / "test",
                max_images=int(PRED_EXPORT_MAX_IMAGES),
                save_input=bool(PRED_EXPORT_SAVE_INPUT),
                save_gt=bool(PRED_EXPORT_SAVE_GT),
            )
        except Exception as e:
            print(f"! Prediction export failed for {model_name}: {e}")

    # Optional: paper-standard boundary matching via pdollar/edges (BSDS eval)
    bsds_metrics = None
    if config.get('eval_protocol', EVAL_PROTOCOL) == 'bsds':
        try:
            eval_root = variant_dir / 'pdollar_eval'
            res_dir = eval_root / 'res'
            gt_dir = eval_root / 'groundTruth'
            out_dir = eval_root / 'out'
            export_pred_pngs_for_edges(test_preds, test_names, res_dir)
            write_bsds_groundtruth_mats_from_arrays(test_labels, test_names, gt_dir)
            run_pdollar_edges_eval(res_dir=res_dir, gt_dir=gt_dir, out_dir=out_dir)
            bsds_metrics = parse_pdollar_eval_bdry(out_dir)
            bsds_path = variant_dir / "test_metrics_pdollar_edges.json"
            bsds_path.write_text(json.dumps(bsds_metrics, indent=2), encoding="utf-8")
            _backup_put(bsds_path)
        except Exception as e:
            print(f"! BSDS eval requested but failed: {e}")
            bsds_metrics = None

    final_loss = epoch_losses[-1] if epoch_losses else float('nan')

    print(f"✓ Test proxy metrics: ODS={final_ods:.4f} | OIS={final_ois:.4f} | AP={final_ap:.4f}")
    if bsds_metrics is not None:
        print(f"✓ Test pdollar/edges metrics: ODS={bsds_metrics.get('ODS', float('nan')):.4f} | OIS={bsds_metrics.get('OIS', float('nan')):.4f} | AP={bsds_metrics.get('AP', float('nan')):.4f}")
    print(f"✓ Best epoch (by proxy ODS): {best_epoch} | Training time: {elapsed:.0f}s")

    # Return pdollar metrics if available; otherwise proxy
    if bsds_metrics is not None:
        return float(bsds_metrics.get('ODS', final_ods)), float(bsds_metrics.get('OIS', final_ois)), float(bsds_metrics.get('AP', final_ap)), final_loss, best_epoch, elapsed, epoch_losses
    return final_ods, final_ois, final_ap, final_loss, best_epoch, elapsed, epoch_losses

print("✓ Training function ready (saves checkpoints/.pt + exports test predictions)")

✓ Training function ready (EdgeLoss + correct eval shapes + ELC supported)


In [ ]:
# Cell 7: RUN ABLATION STUDY (Main Loop)
print(f"\n{'='*80}")
print("STARTING ABLATION STUDY - Running all {0} variants".format(len(EXPERIMENTS)))
print(f"{'='*80}\n")

study_start = time.time()
failed_experiments = []
epoch_histories = {}  # Track training curves

for i, exp in enumerate(EXPERIMENTS, 1):
    exp_name = exp['name']
    config = {k: v for k, v in exp.items() if k not in ['name', 'description']}
    
    print(f"\n[{i}/{len(EXPERIMENTS)}] Training: {exp_name}")
    
    try:
        # Call actual training function
        ods, ois, ap, train_loss, best_epoch, elapsed, epoch_losses = train_variant(
            exp_name, config, epochs=EPOCHS_PER_VARIANT
        )
        
        # Log result
        tracker.log_experiment(exp_name, config, ods, ois, ap, train_loss, best_epoch, elapsed)
        
        # Store epoch history
        epoch_histories[exp_name] = epoch_losses
        
    except Exception as e:
        print(f"✗ FAILED: {e}")
        failed_experiments.append((exp_name, str(e)))

study_elapsed = time.time() - study_start

print(f"\n\n{'='*80}")
print("ABLATION STUDY COMPLETE")
print(f"{'='*80}")
print(f"Total variants: {len(EXPERIMENTS)}")
print(f"Successful: {len(tracker.results)}")
print(f"Failed: {len(failed_experiments)}")
print(f"Total time: {study_elapsed/60:.1f} min")

if failed_experiments:
    print(f"\nFailed experiments:")
    for name, err in failed_experiments:
        print(f"  - {name}: {err}")



STARTING ABLATION STUDY - Running all 36 variants


[1/36] Training: rcf_baseline

Variant: rcf_baseline
Epochs: 5
Config: {'decoder': 'rcf'}


  Epoch 1: Loss=0.5880 | ODS=0.1416 | OIS=0.1395 | AP=0.0397


  Epoch 2: Loss=0.4562 | ODS=0.1729 | OIS=0.1676 | AP=0.0521


Epoch 3/5:  26%|█████████████████▏                                                 | 129/504 [29:03<1:24:13, 13.47s/it]

In [ ]:
# Cell 8: RESULTS ANALYSIS & SUMMARIES

# Save CSV
df_results = tracker.save_csv()

# Print summaries
tracker.summary_table(top_n=10)

In [ ]:
# Cell 9: PLOT COMPARISONS

# ODS comparison
df_ods_top = tracker.plot_comparison('ODS', top_n=15)

# OIS comparison
df_ois_top = tracker.plot_comparison('OIS', top_n=15)

# AP comparison
df_ap_top = tracker.plot_comparison('AP', top_n=15)

In [ ]:
# Cell 10: CONTRIBUTION ANALYSIS
# Measure impact of each component

df = pd.DataFrame(tracker.results)

print(f"\n{'='*80}")
print("COMPONENT IMPACT ANALYSIS")
print(f"{'='*80}\n")

# Baseline
baseline = df[df['experiment'] == 'rcf_baseline']['ODS'].values[0]
print(f"Baseline (rcf_baseline) ODS: {baseline:.4f}\n")

# === DECODER IMPACT ===
print("1. DECODER IMPACT")
print("-" * 40)
decoders = df[df['disable_pathways'] == 'none']
for decoder in ['rcf', 'elc']:
    dec_df = decoders[decoders['decoder'] == decoder]
    if len(dec_df) > 0:
        avg_ods = dec_df['ODS'].mean()
        delta = avg_ods - baseline
        print(f"  {decoder.upper():5s}: {avg_ods:.4f} (Δ {delta:+.4f})")

# === ENCODER STAGE ABLATIONS ===
print(f"\n2. ENCODER STAGE REMOVALS")
print("-" * 40)
for stage in ['s1', 's2', 's3', 's4']:
    stage_rows = df[df['disable_stages'] == stage]
    if len(stage_rows) > 0:
        avg_ods = stage_rows['ODS'].mean()
        delta = baseline - avg_ods  # Negative = penalty
        print(f"  Remove {stage}: ODS {avg_ods:.4f} (penalty: {delta:+.4f})")

# === PATHWAY ABLATIONS ===
print(f"\n3. XYW PATHWAY ABLATIONS")
print("-" * 40)
for pathway in ['X', 'Y', 'W']:
    path_rows = df[df['disable_pathways'] == pathway]
    if len(path_rows) > 0:
        avg_ods = path_rows['ODS'].mean()
        delta = baseline - avg_ods
        print(f"  Remove {pathway}: ODS {avg_ods:.4f} (penalty: {delta:+.4f})")

# === ARCHITECTURE COMPONENTS ===
print(f"\n4. ARCHITECTURE COMPONENTS")
print("-" * 40)

# PDC vs CV
pdc_rows = df[df['pdc_type'] == '2sd']
cv_rows = df[df['pdc_type'] == 'cv']
if len(pdc_rows) > 0 and len(cv_rows) > 0:
    pdc_avg = pdc_rows['ODS'].mean()
    cv_avg = cv_rows['ODS'].mean()
    print(f"  PDC (2sd): {pdc_avg:.4f}")
    print(f"  Standard (cv): {cv_avg:.4f} (Δ {cv_avg - pdc_avg:+.4f})")

# Norm types
print(f"\n  Normalization:")
for norm_type in ['instance', 'batch', 'group']:
    norm_rows = df[df['norm_type'] == norm_type]
    if len(norm_rows) > 0:
        avg_ods = norm_rows['ODS'].mean()
        print(f"    {norm_type.capitalize():8s}: {avg_ods:.4f}")

# Adaptive gating
gate_on = df[df['disable_adap_gate'] == False]
gate_off = df[df['disable_adap_gate'] == True]
if len(gate_on) > 0 and len(gate_off) > 0:
    print(f"\n  Adaptive Gating:")
    print(f"    Enabled: {gate_on['ODS'].mean():.4f}")
    print(f"    Disabled: {gate_off['ODS'].mean():.4f}")

# === LOSS & EVAL ===
print(f"\n5. LOSS & EVALUATION")
print("-" * 40)

# Dice variations
print(f"  Dice coefficient:")
for dice in [0.0, 0.05, 0.1, 0.2]:
    dice_rows = df[df['dice_coeff'] == dice]
    if len(dice_rows) > 0:
        avg_ods = dice_rows['ODS'].mean()
        print(f"    {dice:.2f}: {avg_ods:.4f}")

# CE positive weighting
print(f"\n  CE Positive Weight:")
for weight in [1.0, 2.0, 4.0]:
    weight_rows = df[df['ce_pos_weight'] == weight]
    if len(weight_rows) > 0:
        avg_ods = weight_rows['ODS'].mean()
        print(f"    {weight:.1f}x: {avg_ods:.4f}")

print(f"\n{'='*80}")

In [ ]:
# Cell 10.5: ADVANCED VISUALIZATIONS (Scatter Plots, Heatmaps, Distributions)

df = pd.DataFrame(tracker.results)

# 1. SCATTER: ODS vs AP (correlation analysis)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ODS vs AP
ax = axes[0, 0]
scatter = ax.scatter(df['ODS'], df['AP'], c=df['train_loss'], cmap='RdYlGn_r', s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('ODS', fontsize=11, fontweight='bold')
ax.set_ylabel('AP', fontsize=11, fontweight='bold')
ax.set_title('ODS vs AP (colored by training loss)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Loss')
ax.grid(alpha=0.3)

# ODS vs Training Time
ax = axes[0, 1]
scatter = ax.scatter(df['ODS'], df['time_sec'], c=df['OIS'], cmap='viridis', s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('ODS', fontsize=11, fontweight='bold')
ax.set_ylabel('Training Time (s)', fontsize=11, fontweight='bold')
ax.set_title('ODS vs Training Time (colored by OIS)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='OIS')
ax.grid(alpha=0.3)

# ODS vs Loss
ax = axes[1, 0]
scatter = ax.scatter(df['ODS'], df['train_loss'], c=df['best_epoch'], cmap='cool', s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('ODS', fontsize=11, fontweight='bold')
ax.set_ylabel('Final Training Loss', fontsize=11, fontweight='bold')
ax.set_title('ODS vs Training Loss (colored by best epoch)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Best Epoch')
ax.grid(alpha=0.3)

# OIS vs AP
ax = axes[1, 1]
scatter = ax.scatter(df['OIS'], df['AP'], c=df['time_sec'], cmap='plasma', s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('OIS', fontsize=11, fontweight='bold')
ax.set_ylabel('AP', fontsize=11, fontweight='bold')
ax.set_title('OIS vs AP (colored by time)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Time (s)')
ax.grid(alpha=0.3)

plt.tight_layout()
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = ABLATION_DIR / f"scatter_analysis_{timestamp}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved scatter plots: {plot_path}")
plt.show()


In [ ]:
# Cell 10.6: COMPONENT-SPECIFIC ANALYSIS PLOTS

df = pd.DataFrame(tracker.results)
baseline_ods = df[df['experiment'] == 'rcf_baseline']['ODS'].values[0]

# Figure 1: Component impact by category
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Decoder Impact
ax = axes[0, 0]
decoder_impact = df.groupby('decoder')['ODS'].mean().reset_index()
decoder_impact['impact'] = decoder_impact['ODS'] - baseline_ods
colors_dec = ['green' if x > 0 else 'red' for x in decoder_impact['impact']]
ax.barh(decoder_impact['decoder'], decoder_impact['impact'], color=colors_dec, alpha=0.7, edgecolor='black')
ax.set_xlabel('ODS Change vs Baseline', fontweight='bold')
ax.set_title('Decoder Impact', fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(alpha=0.3, axis='x')

# 2. Encoder Stage Removal Impact
ax = axes[0, 1]
stages_to_test = ['s1', 's2', 's3', 's4']
stage_impact = []
for stage in stages_to_test:
    stage_rows = df[df['disable_stages'] == stage]
    if len(stage_rows) > 0:
        impact = baseline_ods - stage_rows['ODS'].mean()
        stage_impact.append({'stage': stage, 'penalty': impact})

stage_df = pd.DataFrame(stage_impact)
colors_stage = ['red' if x > 0 else 'green' for x in stage_df['penalty']]
ax.barh(stage_df['stage'], stage_df['penalty'], color=colors_stage, alpha=0.7, edgecolor='black')
ax.set_xlabel('ODS Penalty (drop)', fontweight='bold')
ax.set_title('Encoder Stage Importance', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='x')

# 3. XYW Pathway Impact
ax = axes[0, 2]
pathways_to_test = ['X', 'Y', 'W']
pathway_impact = []
for pathway in pathways_to_test:
    path_rows = df[df['disable_pathways'] == pathway]
    if len(path_rows) > 0:
        impact = baseline_ods - path_rows['ODS'].mean()
        pathway_impact.append({'pathway': pathway, 'penalty': impact})

pathway_df = pd.DataFrame(pathway_impact)
colors_path = ['red' if x > 0 else 'green' for x in pathway_df['penalty']]
ax.barh(pathway_df['pathway'], pathway_df['penalty'], color=colors_path, alpha=0.7, edgecolor='black')
ax.set_xlabel('ODS Penalty (drop)', fontweight='bold')
ax.set_title('XYW Pathway Importance', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='x')

# 4. Normalization Impact
ax = axes[1, 0]
norm_types = df[df['norm_type'].isin(['instance', 'batch', 'group'])]['norm_type'].unique()
norm_impact = df[df['norm_type'].isin(norm_types)].groupby('norm_type')['ODS'].mean().reset_index()
norm_impact['impact'] = norm_impact['ODS'] - baseline_ods
colors_norm = ['green' if x > 0 else 'red' for x in norm_impact['impact']]
ax.barh(norm_impact['norm_type'], norm_impact['impact'], color=colors_norm, alpha=0.7, edgecolor='black')
ax.set_xlabel('ODS Change vs Baseline', fontweight='bold')
ax.set_title('Normalization Type Impact', fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(alpha=0.3, axis='x')

# 5. Learnable Deconv Impact
ax = axes[1, 1]
deconv_variants = [
    ('Frozen (default)', df[df['learnable_deconv'] == False]['ODS'].mean()),
    ('Learnable', df[df['learnable_deconv'] == True]['ODS'].mean()),
]
deconv_names, deconv_vals = zip(*deconv_variants)
deconv_impacts = [v - baseline_ods for v in deconv_vals]
colors_deconv = ['green' if x > 0 else 'red' for x in deconv_impacts]
ax.barh(deconv_names, deconv_impacts, color=colors_deconv, alpha=0.7, edgecolor='black')
ax.set_xlabel('ODS Change vs Baseline', fontweight='bold')
ax.set_title('Deconv Learnable Impact', fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(alpha=0.3, axis='x')

# 6. PDC vs Standard Conv
ax = axes[1, 2]
pdc_types = df['pdc_type'].unique()
pdc_impact = []
for pdc in pdc_types:
    pdc_rows = df[df['pdc_type'] == pdc]
    if len(pdc_rows) > 0:
        avg_ods = pdc_rows['ODS'].mean()
        impact = avg_ods - baseline_ods
        pdc_impact.append({'type': f"{pdc} (2sd)" if pdc == '2sd' else 'Standard Conv', 'impact': impact})

pdc_df = pd.DataFrame(pdc_impact)
colors_pdc = ['green' if x > 0 else 'red' for x in pdc_df['impact']]
ax.barh(pdc_df['type'], pdc_df['impact'], color=colors_pdc, alpha=0.7, edgecolor='black')
ax.set_xlabel('ODS Change vs Baseline', fontweight='bold')
ax.set_title('PDC vs Standard Conv Impact', fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = ABLATION_DIR / f"component_impact_{timestamp}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved component impact plots: {plot_path}")
plt.show()


In [ ]:
# Cell 10.7: LEARNING CURVES & DISTRIBUTION ANALYSIS

df = pd.DataFrame(tracker.results)

# 1. Learning curves for top 5 variants
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 5 by ODS
top_5 = df.nlargest(5, 'ODS')['experiment'].tolist()

ax = axes[0]
for exp_name in top_5:
    if exp_name in epoch_histories:
        losses = epoch_histories[exp_name]
        ax.plot(range(1, len(losses)+1), losses, marker='o', label=exp_name, linewidth=2)

ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax.set_ylabel('Training Loss', fontsize=11, fontweight='bold')
ax.set_title('Learning Curves: Top 5 Variants by ODS', fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)

# Bottom 5 by ODS (worst)
bottom_5 = df.nsmallest(5, 'ODS')['experiment'].tolist()

ax = axes[1]
for exp_name in bottom_5:
    if exp_name in epoch_histories:
        losses = epoch_histories[exp_name]
        ax.plot(range(1, len(losses)+1), losses, marker='x', label=exp_name, linewidth=2)

ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax.set_ylabel('Training Loss', fontsize=11, fontweight='bold')
ax.set_title('Learning Curves: Worst 5 Variants (Bottom ODS)', fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = ABLATION_DIR / f"learning_curves_{timestamp}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved learning curves: {plot_path}")
plt.show()

# 2. Distribution analysis by component
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Decoder distribution
ax = axes[0, 0]
for decoder in df['decoder'].unique():
    data = df[df['decoder'] == decoder]['ODS']
    ax.hist(data, alpha=0.6, label=decoder, bins=5, edgecolor='black')
ax.set_xlabel('ODS', fontweight='bold')
ax.set_ylabel('Count', fontweight='bold')
ax.set_title('ODS Distribution by Decoder', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Normalization distribution
ax = axes[0, 1]
for norm in df[df['norm_type'].isin(['instance', 'batch', 'group'])]['norm_type'].unique():
    data = df[df['norm_type'] == norm]['ODS']
    ax.hist(data, alpha=0.6, label=norm, bins=5, edgecolor='black')
ax.set_xlabel('ODS', fontweight='bold')
ax.set_ylabel('Count', fontweight='bold')
ax.set_title('ODS Distribution by Norm Type', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# PDC vs Conv distribution
ax = axes[0, 2]
for pdc in df['pdc_type'].unique():
    label = f"{pdc} (2sd)" if pdc == '2sd' else 'Standard (cv)'
    data = df[df['pdc_type'] == pdc]['ODS']
    ax.hist(data, alpha=0.6, label=label, bins=5, edgecolor='black')
ax.set_xlabel('ODS', fontweight='bold')
ax.set_ylabel('Count', fontweight='bold')
ax.set_title('ODS Distribution by Conv Type', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Training loss distribution
ax = axes[1, 0]
ax.hist(df['train_loss'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Training Loss', fontweight='bold')
ax.set_ylabel('Count', fontweight='bold')
ax.set_title('Training Loss Distribution (All Variants)', fontsize=12, fontweight='bold')
ax.axvline(df['train_loss'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["train_loss"].mean():.4f}')
ax.legend()
ax.grid(alpha=0.3)

# Training time distribution
ax = axes[1, 1]
ax.hist(df['time_sec'], bins=15, color='forestgreen', edgecolor='black', alpha=0.7)
ax.set_xlabel('Training Time (s)', fontweight='bold')
ax.set_ylabel('Count', fontweight='bold')
ax.set_title('Training Time Distribution', fontsize=12, fontweight='bold')
ax.axvline(df['time_sec'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["time_sec"].mean():.0f}s')
ax.legend()
ax.grid(alpha=0.3)

# Convergence speed (best_epoch distribution)
ax = axes[1, 2]
ax.hist(df['best_epoch'], bins=range(1, int(df['best_epoch'].max())+2), color='orange', edgecolor='black', alpha=0.7)
ax.set_xlabel('Best Epoch', fontweight='bold')
ax.set_ylabel('Count', fontweight='bold')
ax.set_title('Convergence Speed (Best Epoch Distribution)', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = ABLATION_DIR / f"distributions_{timestamp}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved distribution plots: {plot_path}")
plt.show()


In [ ]:
# Cell 10.8: HEATMAP & CONTRIBUTION RANKING

df = pd.DataFrame(tracker.results)
baseline_ods = df[df['experiment'] == 'rcf_baseline']['ODS'].values[0]

# 1. Contribution analysis - compute impact of each component ablation
contributions = []

# Stages
for stage in ['s1', 's2', 's3', 's4']:
    stage_rows = df[df['disable_stages'] == stage]
    if len(stage_rows) > 0:
        penalty = baseline_ods - stage_rows['ODS'].mean()
        contributions.append({'component': f'No {stage}', 'impact': penalty, 'type': 'Encoder'})

# Pathways
for pathway in ['X', 'Y', 'W']:
    path_rows = df[df['disable_pathways'] == pathway]
    if len(path_rows) > 0:
        penalty = baseline_ods - path_rows['ODS'].mean()
        contributions.append({'component': f'No {pathway}', 'impact': penalty, 'type': 'Pathway'})

# Architecture
gate_rows = df[df['disable_adap_gate'] == True]
if len(gate_rows) > 0:
    penalty = baseline_ods - gate_rows['ODS'].mean()
    contributions.append({'component': 'No AdapGate', 'impact': penalty, 'type': 'Architecture'})

short_rows = df[df['disable_shortcuts'] == True]
if len(short_rows) > 0:
    penalty = baseline_ods - short_rows['ODS'].mean()
    contributions.append({'component': 'No Shortcuts', 'impact': penalty, 'type': 'Architecture'})

# Conv type
cv_rows = df[df['pdc_type'] == 'cv']
if len(cv_rows) > 0:
    penalty = baseline_ods - cv_rows['ODS'].mean()
    contributions.append({'component': 'Std Conv', 'impact': penalty, 'type': 'Architecture'})

contrib_df = pd.DataFrame(contributions).sort_values('impact', ascending=False)

# Plot 1: Contribution ranking
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

ax = axes[0]
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(contrib_df)))
bars = ax.barh(contrib_df['component'], contrib_df['impact'], color=colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('ODS Drop (Importance Score)', fontsize=11, fontweight='bold')
ax.set_title('Component Importance Ranking\n(Higher = More Critical)', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(alpha=0.3, axis='x')

# Add value labels
for i, (idx, row) in enumerate(contrib_df.iterrows()):
    ax.text(row['impact'] + 0.002, i, f"{row['impact']:.4f}", va='center', fontweight='bold', fontsize=9)

# Plot 2: Contribution by type
ax = axes[1]
contrib_by_type = contrib_df.groupby('type')['impact'].sum().sort_values(ascending=False)
colors_type = plt.cm.Set3(np.linspace(0, 1, len(contrib_by_type)))
ax.bar(contrib_by_type.index, contrib_by_type.values, color=colors_type, edgecolor='black', linewidth=1.5, alpha=0.8)
ax.set_ylabel('Total ODS Drop', fontsize=11, fontweight='bold')
ax.set_title('Importance by Component Type', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

# Add value labels
for i, (comp_type, val) in enumerate(contrib_by_type.items()):
    ax.text(i, val + 0.005, f"{val:.4f}", ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = ABLATION_DIR / f"contribution_ranking_{timestamp}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved contribution ranking: {plot_path}")
plt.show()

# 2. Create heatmap of metric correlations
fig, ax = plt.subplots(figsize=(10, 8))
metrics = ['ODS', 'OIS', 'AP', 'train_loss', 'time_sec', 'best_epoch']
corr_matrix = df[metrics].corr()

import matplotlib.patches as mpatches
im = ax.imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)

ax.set_xticks(range(len(metrics)))
ax.set_yticks(range(len(metrics)))
ax.set_xticklabels(metrics, rotation=45, ha='right')
ax.set_yticklabels(metrics)
ax.set_title('Metric Correlation Heatmap', fontsize=12, fontweight='bold')

# Add correlation values
for i in range(len(metrics)):
    for j in range(len(metrics)):
        text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                      ha="center", va="center", color="black" if abs(corr_matrix.iloc[i, j]) < 0.5 else "white",
                      fontweight='bold', fontsize=10)

plt.colorbar(im, ax=ax, label='Correlation')
plt.tight_layout()
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = ABLATION_DIR / f"correlation_heatmap_{timestamp}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved correlation heatmap: {plot_path}")
plt.show()

print(f"\n{'='*80}")
print("KEY INSIGHTS FROM CONTRIBUTION ANALYSIS:")
print(f"{'='*80}")
print(contrib_df.to_string(index=False))


In [ ]:
# Cell 11: DETAILED EXPORT & DOCUMENTATION

# Export detailed JSON
json_path = ABLATION_DIR / f"ablation_detailed_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(json_path, 'w') as f:
    json.dump({
        'study_config': {
            'dataset': DATA_ROOT,
            'epochs_per_variant': EPOCHS_PER_VARIANT,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
        },
        'results': tracker.results,
    }, f, indent=2)
print(f"✓ Saved detailed JSON: {json_path}")

# Export experiment registry
exp_path = ABLATION_DIR / f"experiments_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(exp_path, 'w') as f:
    json.dump(EXPERIMENTS, f, indent=2)
print(f"✓ Saved experiment registry: {exp_path}")

print(f"\n✓ All results saved to: {ABLATION_DIR}")
print(f"  - CSV results")
print(f"  - Comparison plots (ODS/OIS/AP)")
print(f"  - Detailed JSON")
print(f"  - Experiment registry")

In [ ]:
# Cell 12: RECOMMENDATIONS & SUMMARY

df = pd.DataFrame(tracker.results)

print(f"\n{'='*80}")
print("KEY FINDINGS & RECOMMENDATIONS")
print(f"{'='*80}\n")

# Best variant
best_idx = df['ODS'].idxmax()
best_row = df.loc[best_idx]
print(f"🏆 BEST VARIANT: {best_row['experiment']}")
print(f"   ODS: {best_row['ODS']:.4f}, OIS: {best_row['OIS']:.4f}, AP: {best_row['AP']:.4f}")
print(f"   Config: {best_row.to_dict()}\n")

# Most critical components (largest penalty when removed)
print(f"\n⚠️  MOST CRITICAL COMPONENTS (removing hurts most):")
baseline = df[df['experiment'] == 'rcf_baseline']['ODS'].values[0]

impact = []
# Stages
for stage in ['s1', 's2', 's3', 's4']:
    stage_rows = df[df['disable_stages'] == stage]
    if len(stage_rows) > 0:
        penalty = baseline - stage_rows['ODS'].mean()
        impact.append((f"Remove {stage}", penalty))

# Pathways
for pathway in ['X', 'Y', 'W']:
    path_rows = df[df['disable_pathways'] == pathway]
    if len(path_rows) > 0:
        penalty = baseline - path_rows['ODS'].mean()
        impact.append((f"Remove {pathway}", penalty))

impact.sort(key=lambda x: x[1], reverse=True)
for comp, penalty in impact[:5]:
    print(f"   {comp:20s}: {penalty:+.4f} ODS drop")

print(f"\n{'='*80}")

In [ ]:
# Cell 13: PREDICTION VISUALIZATION - Top Variants on Test Images

import matplotlib.gridspec as gridspec

# Get top 5 variants
df = pd.DataFrame(tracker.results)
top_5_variants = df.nlargest(5, 'ODS')['experiment'].tolist()

# Load one test batch
test_batch = next(iter(test_loader))
test_images = test_batch['images'][:4]  # First 4 images
test_labels = test_batch['labels'][:4]
filenames = test_batch['filename'][:4]

# Create comprehensive visualization
fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(4, 6, figure=fig, hspace=0.4, wspace=0.3)

# For each test image
for img_idx in range(4):
    img = test_images[img_idx]
    gt = test_labels[img_idx]
    
    # Denormalize image
    img_vis = img.cpu().numpy().transpose(1, 2, 0)
    img_vis = (img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    
    # Show input image
    ax = fig.add_subplot(gs[img_idx, 0])
    ax.imshow(img_vis)
    ax.set_title(f'Input: {filenames[img_idx]}', fontweight='bold', fontsize=10)
    ax.axis('off')
    
    # Show ground truth
    ax = fig.add_subplot(gs[img_idx, 1])
    ax.imshow(gt.cpu().squeeze(), cmap='gray')
    ax.set_title('Ground Truth', fontweight='bold', fontsize=10)
    ax.axis('off')
    
    # Show predictions from top 5 variants
    for var_idx, variant_name in enumerate(top_5_variants):
        ax = fig.add_subplot(gs[img_idx, var_idx + 2])
        
        try:
            # Load model
            exp_config = next((e for e in EXPERIMENTS if e['name'] == variant_name), None)
            if exp_config:
                config = {k: v for k, v in exp_config.items() if k not in ['name', 'description']}
                model = XYWNet(config).to(DEVICE)
                
                # Load best checkpoint
                checkpoint_path = MODELS_DIR / f"{variant_name}_epoch*.pth"
                import glob
                checkpoints = glob.glob(str(checkpoint_path))
                if checkpoints:
                    latest_checkpoint = max(checkpoints, key=os.path.getctime)
                    model.load_state_dict(torch.load(latest_checkpoint, map_location=DEVICE))
                
                model.eval()
                with torch.no_grad():
                    pred = model(img.unsqueeze(0).to(DEVICE))
                    pred = pred.cpu().squeeze().numpy()
                
                ax.imshow(pred, cmap='gray', vmin=0, vmax=1)
                ods_score = df[df['experiment'] == variant_name]['ODS'].values[0]
                ax.set_title(f'{variant_name}\nODS: {ods_score:.4f}', fontweight='bold', fontsize=9)
        except Exception as e:
            ax.text(0.5, 0.5, f'Error:\n{str(e)[:30]}', ha='center', va='center')
        
        ax.axis('off')

plt.suptitle('Ablation Study: Top 5 Variants - Visual Comparison', fontsize=14, fontweight='bold', y=0.995)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = ABLATION_DIR / f"prediction_comparison_top5_{timestamp}.png"
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
print(f"✓ Saved prediction comparison: {plot_path}")
plt.show()


In [ ]:
# Cell 14: ENCODER STAGE VISUALIZATION - Intermediate Outputs

# Modified XYWNet to return intermediate features
class XYWNetWithIntermediates(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.base_model = XYWNet(config)
        self.encode = self.base_model.encode
        self.decode = self.base_model.decode
    
    def forward(self, x):
        # Get intermediate outputs
        s1, s2, s3, s4 = self.encode(x)
        return s1, s2, s3, s4, self.decode([s1, s2, s3, s4])

# Visualize best variant's encoder stages
best_variant = df.nlargest(1, 'ODS')['experiment'].values[0]
print(f"Visualizing encoder stages for: {best_variant}")

# Load model
exp_config = next((e for e in EXPERIMENTS if e['name'] == best_variant), None)
if exp_config:
    config = {k: v for k, v in exp_config.items() if k not in ['name', 'description']}
    model_with_inter = XYWNetWithIntermediates(config).to(DEVICE)
    
    # Load checkpoint
    checkpoint_path = MODELS_DIR / f"{best_variant}_epoch*.pth"
    import glob
    checkpoints = glob.glob(str(checkpoint_path))
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=os.path.getctime)
        # Need to load into base_model
        state_dict = torch.load(latest_checkpoint, map_location=DEVICE)
        model_with_inter.base_model.load_state_dict(state_dict)
    
    model_with_inter.eval()
    
    # Get one test image
    test_batch = next(iter(test_loader))
    test_img = test_batch['images'][0:1].to(DEVICE)
    test_gt = test_batch['labels'][0:1]
    
    with torch.no_grad():
        s1_out, s2_out, s3_out, s4_out, final_pred = model_with_inter(test_img)
    
    # Visualize encoder stages
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Input
    img_vis = test_img[0].cpu().numpy().transpose(1, 2, 0)
    img_vis = (img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    axes[0, 0].imshow(img_vis)
    axes[0, 0].set_title('Input Image', fontweight='bold', fontsize=11)
    axes[0, 0].axis('off')
    
    # Ground truth
    axes[0, 1].imshow(test_gt[0].cpu().squeeze(), cmap='gray')
    axes[0, 1].set_title('Ground Truth', fontweight='bold', fontsize=11)
    axes[0, 1].axis('off')
    
    # Final prediction
    axes[0, 2].imshow(final_pred[0].cpu().squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[0, 2].set_title('Final Prediction', fontweight='bold', fontsize=11)
    axes[0, 2].axis('off')
    
    # Encoder stages
    stage_outputs = [s1_out, s2_out, s3_out, s4_out]
    stage_names = ['Stage 1 (s1)', 'Stage 2 (s2)', 'Stage 3 (s3)', 'Stage 4 (s4)']
    
    for stage_idx, (stage_out, stage_name) in enumerate(zip(stage_outputs, stage_names)):
        # Visualize mean across channels
        stage_mean = stage_out[0].mean(dim=0).cpu().numpy()
        
        # Normalize to 0-1 for visualization
        stage_vis = (stage_mean - stage_mean.min()) / (stage_mean.max() - stage_mean.min() + 1e-8)
        
        ax = axes[1, stage_idx] if stage_idx < 3 else axes[0, stage_idx - 2]
        if stage_idx == 3:
            ax = axes[1, 2]
        
        im = ax.imshow(stage_vis, cmap='hot')
        ax.set_title(f'{stage_name}\nShape: {tuple(stage_out.shape[1:])}', fontweight='bold', fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.suptitle(f'Encoder Stages Visualization: {best_variant}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    plot_path = ABLATION_DIR / f"encoder_stages_{best_variant}_{timestamp}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved encoder stages visualization: {plot_path}")
    plt.show()


In [ ]:
# Cell 15: DECODER REFINEMENT STAGES - Upsampling Progression

# Create modified decoder to capture intermediate refinement stages
class DecoderWithStages(nn.Module):
    def __init__(self, base_decoder):
        super().__init__()
        self.f43 = base_decoder.f43
        self.f32 = base_decoder.f32
        self.f21 = base_decoder.f21
        self.f = base_decoder.f
    
    def forward(self, end_points):
        s1, s2, s3, s4 = end_points
        
        # Decoder stages
        s3_refined = self.f43(s2, s4)  # Fuse s4 with s2
        s2_refined = self.f32(s1, s3_refined)  # Fuse s3_refined with s1
        s1_refined = self.f21(s1, s2_refined)  # Final refinement
        
        # Output predictions at each stage
        out_s4 = self.f(s4)
        out_s3 = self.f(s3_refined)
        out_s2 = self.f(s2_refined)
        out_s1 = self.f(s1_refined)
        
        return {
            's4_refined': out_s4,
            's3_refined': out_s3,
            's2_refined': out_s2,
            's1_refined': out_s1,
            'final': out_s1
        }

# Test with best variant
if exp_config:
    model_full = XYWNet(config).to(DEVICE)
    
    # Load checkpoint
    checkpoint_path = MODELS_DIR / f"{best_variant}_epoch*.pth"
    checkpoints = glob.glob(str(checkpoint_path))
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=os.path.getctime)
        model_full.load_state_dict(torch.load(latest_checkpoint, map_location=DEVICE))
    
    model_full.eval()
    
    # Create decoder wrapper
    decoder_stages = DecoderWithStages(model_full.decode)
    
    # Get intermediate encoder outputs
    test_batch = next(iter(test_loader))
    test_img = test_batch['images'][0:1].to(DEVICE)
    test_gt = test_batch['labels'][0:1]
    
    with torch.no_grad():
        end_points = model_full.encode(test_img)
        decoder_outputs = decoder_stages(end_points)
    
    # Visualize decoder refinement
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Input
    img_vis = test_img[0].cpu().numpy().transpose(1, 2, 0)
    img_vis = (img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    axes[0, 0].imshow(img_vis)
    axes[0, 0].set_title('Input Image', fontweight='bold', fontsize=11)
    axes[0, 0].axis('off')
    
    # Ground truth
    axes[0, 1].imshow(test_gt[0].cpu().squeeze(), cmap='gray')
    axes[0, 1].set_title('Ground Truth', fontweight='bold', fontsize=11)
    axes[0, 1].axis('off')
    
    # Final output
    axes[0, 2].imshow(decoder_outputs['final'][0].cpu().squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[0, 2].set_title('Final Output', fontweight='bold', fontsize=11)
    axes[0, 2].axis('off')
    
    # Decoder stages
    decoder_stage_keys = ['s4_refined', 's3_refined', 's2_refined', 's1_refined']
    decoder_stage_names = ['After S4→S3 (Coarse)', 'After S3→S2 (Mid)', 'After S2→S1 (Fine)', 'Final Refined']
    
    for stage_idx, (key, name) in enumerate(zip(decoder_stage_keys, decoder_stage_names)):
        row = (stage_idx + 3) // 3
        col = (stage_idx + 3) % 3
        
        pred = decoder_outputs[key][0].cpu().squeeze().numpy()
        im = axes[row, col].imshow(pred, cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(f'{name}\nDecoder Step {stage_idx+1}', fontweight='bold', fontsize=10)
        axes[row, col].axis('off')
    
    # Hide empty subplot
    axes[1, 2].axis('off')
    
    plt.suptitle(f'Decoder Refinement Stages: {best_variant}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    plot_path = ABLATION_DIR / f"decoder_stages_{best_variant}_{timestamp}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved decoder stages visualization: {plot_path}")
    plt.show()


In [ ]:
# Cell 16: ABLATION IMPACT VISUALIZATION - See Effect of Removing Components

import glob

# Choose comparison: baseline vs specific ablations
baseline_name = 'rcf_baseline'

# Get test images
test_batch = next(iter(test_loader))
test_imgs = test_batch['images'][:3].to(DEVICE)
test_gts = test_batch['labels'][:3]

# Groups to compare
ablation_groups = {
    'Encoder Stages': ['rcf_baseline', 'no_s1', 'no_s2', 'no_s3', 'no_s4'],
    'XYW Pathways': ['rcf_baseline', 'no_X', 'no_Y', 'no_W'],
    'Decoder': ['rcf_baseline', 'elc_enabled'],
}

for group_name, variant_list in ablation_groups.items():
    # Filter to variants that exist in results
    variant_list = [v for v in variant_list if v in df['experiment'].values]
    
    if len(variant_list) < 2:
        continue
    
    fig, axes = plt.subplots(len(test_imgs), len(variant_list), figsize=(14, 12))
    
    if len(test_imgs) == 1:
        axes = axes.reshape(1, -1)
    
    # For each test image
    for img_idx, (test_img, test_gt) in enumerate(zip(test_imgs, test_gts)):
        # Show input only on first row
        if img_idx == 0:
            # Add input column
            fig.text(0.02, 0.5 - img_idx*0.3, 'Input', ha='right', va='center', fontsize=9, fontweight='bold')
        
        # For each variant
        for var_idx, variant_name in enumerate(variant_list):
            try:
                # Load model
                exp_config = next((e for e in EXPERIMENTS if e['name'] == variant_name), None)
                if exp_config:
                    config = {k: v for k, v in exp_config.items() if k not in ['name', 'description']}
                    model = XYWNet(config).to(DEVICE)
                    
                    # Load checkpoint
                    checkpoint_path = MODELS_DIR / f"{variant_name}_epoch*.pth"
                    checkpoints = glob.glob(str(checkpoint_path))
                    if checkpoints:
                        latest_checkpoint = max(checkpoints, key=os.path.getctime)
                        model.load_state_dict(torch.load(latest_checkpoint, map_location=DEVICE))
                    
                    model.eval()
                    with torch.no_grad():
                        pred = model(test_img.unsqueeze(0).to(DEVICE))
                        pred = pred.cpu().squeeze().numpy()
                    
                    ax = axes[img_idx, var_idx]
                    im = ax.imshow(pred, cmap='gray', vmin=0, vmax=1)
                    
                    # Title
                    ods_val = df[df['experiment'] == variant_name]['ODS'].values[0]
                    if img_idx == 0:
                        ax.set_title(f'{variant_name}\nODS: {ods_val:.4f}', fontsize=9, fontweight='bold')
                    
                    ax.axis('off')
            except Exception as e:
                ax = axes[img_idx, var_idx]
                ax.text(0.5, 0.5, f'Error', ha='center', va='center', fontsize=8)
                ax.axis('off')
    
    plt.suptitle(f'Ablation Impact: {group_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    plot_path = ABLATION_DIR / f"ablation_comparison_{group_name.replace(' ', '_')}_{timestamp}.png"
    plt.savefig(plot_path, dpi=100, bbox_inches='tight')
    print(f"✓ Saved ablation comparison ({group_name}): {plot_path}")
    plt.show()


In [ ]:
# Cell 17: COMPREHENSIVE PREDICTION GRID - All Info in One View

# Get top 10 variants
df = pd.DataFrame(tracker.results)
top_10 = df.nlargest(10, 'ODS')

# Get test image
test_batch = next(iter(test_loader))
test_img = test_batch['images'][0:1]
test_gt = test_batch['labels'][0:1]

# Denormalize image
img_vis = test_img[0].cpu().numpy().transpose(1, 2, 0)
img_vis = (img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)

# Create figure with subplots
fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.35, wspace=0.25)

# Show input and GT
ax = fig.add_subplot(gs[0, 0])
ax.imshow(img_vis)
ax.set_title('Input Image', fontweight='bold', fontsize=12)
ax.axis('off')

ax = fig.add_subplot(gs[0, 1])
ax.imshow(test_gt[0].cpu().squeeze(), cmap='gray')
ax.set_title('Ground Truth', fontweight='bold', fontsize=12)
ax.axis('off')

ax = fig.add_subplot(gs[0, 2])
ax.axis('off')
ax.text(0.5, 0.5, 'Top 10 Variants\nDetailed Predictions', ha='center', va='center', 
        fontsize=12, fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# Show predictions for top 10
for idx, (_, row) in enumerate(top_10.iterrows()):
    variant_name = row['experiment']
    ax_row = (idx + 3) // 3
    ax_col = (idx + 3) % 3
    
    ax = fig.add_subplot(gs[ax_row, ax_col])
    
    try:
        # Load model
        exp_config = next((e for e in EXPERIMENTS if e['name'] == variant_name), None)
        if exp_config:
            config = {k: v for k, v in exp_config.items() if k not in ['name', 'description']}
            model = XYWNet(config).to(DEVICE)
            
            # Load checkpoint
            checkpoint_path = MODELS_DIR / f"{variant_name}_epoch*.pth"
            checkpoints = glob.glob(str(checkpoint_path))
            if checkpoints:
                latest_checkpoint = max(checkpoints, key=os.path.getctime)
                model.load_state_dict(torch.load(latest_checkpoint, map_location=DEVICE))
            
            model.eval()
            with torch.no_grad():
                pred = model(test_img.to(DEVICE))
                pred = pred.cpu().squeeze().numpy()
            
            im = ax.imshow(pred, cmap='gray', vmin=0, vmax=1)
            
            # Create detailed title with metrics
            title_text = f'{variant_name}\n'
            title_text += f'ODS: {row["ODS"]:.4f} | OIS: {row["OIS"]:.4f} | AP: {row["AP"]:.4f}\n'
            title_text += f'Loss: {row["train_loss"]:.4f} | Epoch: {row["best_epoch"]}'
            
            ax.set_title(title_text, fontsize=9, fontweight='bold', 
                        bbox=dict(boxstyle='round', facecolor='yellow' if idx == 0 else 'white', alpha=0.7))
    except Exception as e:
        ax.text(0.5, 0.5, f'Error loading\nmodel', ha='center', va='center', fontsize=10)
    
    ax.axis('off')

plt.suptitle('Top 10 Variants - Complete Visual Comparison with Metrics', 
             fontsize=14, fontweight='bold', y=0.995)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = ABLATION_DIR / f"comprehensive_top10_{timestamp}.png"
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
print(f"✓ Saved comprehensive top 10 comparison: {plot_path}")
plt.show()

print(f"\n{'='*80}")
print("VISUALIZATION METRICS SUMMARY (Top 10):")
print(f"{'='*80}")
print(top_10[['experiment', 'ODS', 'OIS', 'AP', 'train_loss', 'best_epoch']].to_string(index=False))


## Summary

This ablation study notebook tests **35+ XYW-Net variant configurations** across multiple dimensions:

### What's Tested:
1. **Decoder architectures** (RCF vs ELC)
2. **Encoder stages** (s1, s2, s3, s4 removal)
3. **XYW pathways** (X, Y, W individually and in pairs)
4. **Architecture components** (PDC vs conv, normalization, gating, shortcuts)
5. **Decoder refinement** (learnable vs frozen deconv)
6. **Pooling strategies** (max pool vs stride convolution)
7. **Loss functions** (Dice, positive weighting)
8. **Evaluation controls** (thinning, tolerance)
9. **Combined interactions** (multi-component changes)

### Output:
- **CSV** with results for all variants
- **Bar plots** showing ODS/OIS/AP rankings
- **Component impact analysis** (which parts matter most?)
- **Top/bottom variant summaries**
- **JSON export** for reproducibility

### Next Steps:
1. Replace the placeholder training loop with actual `train_epoch()` from `xywnet_v2.2_gbt.ipynb`
2. Integrate the actual model factory that supports all configuration flags
3. Run 5–10 epochs per variant (adjust EPOCHS_PER_VARIANT)
4. Compare results and identify the **optimal configuration**
5. Train the top 3–5 variants for full 20 epochs to validate